# Training the OV3 sampler arms — energy score, seven arms, one batch stream

Seven paired arms of an 88k-parameter LISA are trained on **identical batches** from one initialisation, so
every difference between them is caused by the arm and nothing else. The question is the one the 8 September
run left open: the sampler wins the proper score and loses the perceptual judges. Is that a fact about
samplers, or about this sampler's distance, its noise pathway, and the statistic read out of it?

Three moves, one model class:

1. **The geometry of $d$.** The energy score $\mathrm{ES}_d(P,y) = \mathbb{E}\,d(Y,y) - \tfrac12 \mathbb{E}\,d(Y,Y')$
   is strictly proper for the law of $\phi(y)$ whenever $d(a,b) = \lVert\phi(a) - \phi(b)\rVert$. Nothing says
   $\phi$ has to be the per-bin log-magnitude the 8 September sampler used. The spectral weight is raised
   tenfold; one arm restricts the waveform term to the low band; two arms add an aggregate-proper term on log
   ERB-band energies, the space ViSQOL's neurogram lives in.
2. **Where the noise enters.** `LISAS` feeds eight Gaussian channels at the 12 kHz input, so every 48 kHz
   output sample inside one input interval is a deterministic function of the same three latents. `LISASD`
   adds four Gaussian channels per *output* sample at the decoder input: noise at the rate of the fine
   structure it has to generate.
3. **The readout.** LSD is squared error in log-power, so its minimiser is the conditional mean of the
   *log*-magnitude, which the waveform ensemble mean does not estimate. The evaluation cells compute it
   directly from the draws.

Training uses the **stacked trainer**: arms of one class share one forward and one backward (parameters carry
a leading arm axis), with bf16 autocast, a compiled decoder MLP, pinned batches, fused Adam, CUDA streams and
no per-step host syncs. The dashboard below reports throughput, **GPU memory** and **per-epoch** statistics live.

## 0. Setup

Packages, device, a persistent cache (Drive on Colab), and seeded random streams. `stream(label)` gives an
independent numpy generator per component, so re-running one cell does not disturb another's draws.

In [ ]:
# ---- inlined verbatim from lisa_rtm.ipynb §0 ----
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for _pkg, _mod in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                   ("soundfile", "soundfile"), ("requests", "requests"), ("tqdm", "tqdm")]:
    ensure(_pkg, _mod)

import os, io, json, math, time, zipfile, hashlib, warnings, dataclasses, shutil, tempfile
from pathlib import Path

import numpy as np
import scipy.signal as sps
import matplotlib.pyplot as plt
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F

IN_COLAB = "google.colab" in sys.modules
print("python  ", sys.version.split()[0])
print("torch   ", torch.__version__)
print("numpy   ", np.__version__)
print("in colab", IN_COLAB)

# ---- device -----------------------------------------------------------------
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("gpu     ", torch.cuda.get_device_name(0))
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("device  ", DEVICE)

# ---- persistent cache -------------------------------------------------------
# Colab disconnects.  Anything expensive (the VCTK subset, checkpoints) goes to Drive if we can
# mount it, so a reconnect costs seconds instead of an hour.
ROOT = None
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/lisa_rtm")
    except Exception as e:
        warnings.warn(f"Drive not mounted ({e}); caching to /content, which a disconnect will lose.")
        ROOT = Path("/content/lisa_rtm")
else:
    ROOT = Path("./lisa_rtm_cache").resolve()

FIXTURES = ROOT / "fixtures"
CKPT     = ROOT / "checkpoints"
FIGS     = ROOT / "figures"
for _d in (FIXTURES, CKPT, FIGS):
    _d.mkdir(parents=True, exist_ok=True)
print("cache   ", ROOT)

# ---- seeding ----------------------------------------------------------------
# One global seed.  Per-component streams are derived by hashing a label, so re-running one
# section does not perturb another section's draws.
SEED = 0

def stream(label, seed=None):
    '''Independent seeded numpy Generator for a named component.'''
    h = hashlib.blake2b(label.encode(), digest_size=8).digest()
    return np.random.default_rng((int.from_bytes(h, "big") ^ (SEED if seed is None else seed)) % (2**63))

def seed_everything(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

seed_everything()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "figure.facecolor": "white"})
print("seeded  ", SEED)

### 0a. Kernel selection — determinism off for **training**, back on for **evaluation**

`seed_everything()` above sets `cudnn.deterministic = True`, `cudnn.benchmark = False` and
`use_deterministic_algorithms(True, warn_only=True)`. That is the right default for a notebook you read
numbers off, and it is very expensive for this model. Under the flag, ATen replaces the fused atomic-add
kernel behind the backward of `torch.gather` (`scatter_add_`) with `_scatter_via_index_put`, which
materialises one int64 key per gathered element — about 147 million keys per arm-draw at batch 32 — and
radix-sorts them. Measured on this repo's own sequential trainer, seven arms, batch 32 × 1 s, A100-80GB:

| setting | ms/step |
| --- | --- |
| deterministic on, `cudnn.benchmark` off (the cell above) | **2261** |
| deterministic off, `cudnn.benchmark` on (`overnight/cell2_trainer.py`) | **587** |

That is 3.9×, for one flag. The seeds, the data order, the noise draws, the jitter draws, the initialisation
and the arm pairing are all untouched, so the seven arms still see identical batches and stay comparable;
what moves is the float reduction order inside the gradient atomics, about $10^{-7}$ relative in fp32, well
under the bf16 rounding the run already carries. The price is run-to-run bitwise reproducibility of the
gradients.

The flag is also mirrored into `torch._inductor.config.deterministic`, which switches Inductor's on-device
autotuning off — so it has to be flipped **before** anything is compiled. It is flipped here, and again in
the stacked trainer's flag block (which sits above its `torch.compile`), so exec order cannot defeat it.

Evaluation puts the strict pair back: §14 and §15 run under `no_grad`, the scatter backward never runs
there, and the sub-pixel decoder's `perturb=False` path has no scatter at all, so determinism costs
essentially nothing on that side.

In [ ]:
# ---- TRAINING kernel selection: see the markdown above (587 vs 2261 ms/step, measured) ----
seed_everything()                     # keep the seeds section 0 set; only the kernel flags change below
torch.use_deterministic_algorithms(False)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
print("training kernels: deterministic", torch.are_deterministic_algorithms_enabled(),
      "| cudnn.benchmark", torch.backends.cudnn.benchmark,
      "| tf32", torch.backends.cuda.matmul.allow_tf32, flush=True)

## 1. Config

One frozen dataclass. `PRESET` picks `FULL` when CUDA is present and `SMOKE` otherwise. The launch cell
overrides batch size and segment length; everything else (model dimensions, learning-rate schedule, gradient
clip, evaluation STFT basis) comes from here.

In [ ]:
# ---- inlined verbatim from lisa_rtm.ipynb §1 ----
@dataclasses.dataclass(frozen=True)
class Config:
    name: str
    fs_hi: int            # target sample rate
    upsample: int         # R:  fs_lo = fs_hi // R
    train_speakers: tuple
    test_speakers: tuple
    utts_per_speaker: int
    seg_samples: int      # training segment length at fs_hi
    # model
    enc_channels: tuple
    enc_kernels: tuple
    dec_hidden: int
    dec_layers: int
    # training
    batch_size: int
    steps: int
    lr: float
    lr_milestones: tuple  # fractions of `steps` at which lr is multiplied by lr_gamma
    lr_gamma: float
    lambda_spec: float    # weight of the multi-scale STFT term.  Official LISA code: 1e-3
    grad_clip: float
    ckpt_every: int
    # analysis
    n_fft: int            # transport / operating basis
    hop: int
    eval_n_fft: int       # evaluation basis (deliberately different from the operating basis)
    eval_hop: int
    n_eval_utts: int
    n_quantiles: int
    lambdas: tuple

    @property
    def fs_lo(self): return self.fs_hi // self.upsample
    @property
    def k_cut(self):
        '''First STFT bin above the input Nyquist, in the transport basis.'''
        return int(np.ceil((self.fs_lo / 2) * self.n_fft / self.fs_hi))
    @property
    def eval_k_cut(self):
        return int(np.ceil((self.fs_lo / 2) * self.eval_n_fft / self.fs_hi))


# Two values below come from the paper / official code (ml-postech/LISA), not from taste:
#   * lambda_spec = 1e-3.  The paper never prints lambda; the released code uses spec_coeff=0.001 and
#     its shipped config is plain L1.  The paper's own ablation shows the spectral term moves LSD by
#     <= 0.01.  An earlier run here used 1.0, which made the phase-blind term 92% of the loss and
#     produced a model with correct magnitudes, random phase, and -5.8 dB SNR (worse than silence).
#   * lr halved at milestones (official code: epochs 10,20,25,30,35,40 of 50), grad clip 1e-3 (stated).
SMOKE = Config(
    name="SMOKE", fs_hi=16000, upsample=4,
    train_speakers=("p225", "p226"), test_speakers=("p236",),
    utts_per_speaker=6, seg_samples=4096,
    enc_channels=(16, 32, 64, 32), enc_kernels=(7, 3, 3, 1),
    dec_hidden=144, dec_layers=5,
    batch_size=8, steps=400, lr=1e-3, lr_milestones=(0.2, 0.4, 0.5, 0.6, 0.7, 0.8), lr_gamma=0.5,
    lambda_spec=1e-3, grad_clip=1e-3, ckpt_every=200,
    n_fft=512, hop=128, eval_n_fft=1024, eval_hop=256,
    n_eval_utts=4, n_quantiles=513, lambdas=(0.0, 0.25, 0.5, 0.75, 1.0),
)

FULL = Config(
    name="FULL", fs_hi=48000, upsample=4,
    train_speakers=("p225", "p226", "p227", "p228", "p229",
                    "p230", "p231", "p232", "p233", "p234"),
    test_speakers=("p236", "p237", "p238"),
    utts_per_speaker=40, seg_samples=12288,
    enc_channels=(16, 32, 64, 32), enc_kernels=(7, 3, 3, 1),
    dec_hidden=144, dec_layers=5,
    batch_size=16, steps=20000, lr=1e-3, lr_milestones=(0.2, 0.4, 0.5, 0.6, 0.7, 0.8), lr_gamma=0.5,
    lambda_spec=1e-3, grad_clip=1e-3, ckpt_every=1000,
    n_fft=2048, hop=512, eval_n_fft=1024, eval_hop=256,
    n_eval_utts=12, n_quantiles=1001,
    lambdas=(0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
)

# ---------------------------------------------------------------------------
PRESET = "FULL" if torch.cuda.is_available() else "SMOKE"     # <-- the one switch
# ---------------------------------------------------------------------------

CFG = {"SMOKE": SMOKE, "FULL": FULL}[PRESET]
RUN = CKPT / f"{CFG.name}_lam{CFG.lambda_spec:g}"     # objective in the path: no silent resumes
RUN.mkdir(parents=True, exist_ok=True)

print(f"preset        {CFG.name}   run dir {RUN.name}")
print(f"rates         {CFG.fs_lo} Hz -> {CFG.fs_hi} Hz  ({CFG.upsample}x)")
print(f"speakers      {len(CFG.train_speakers)} train / {len(CFG.test_speakers)} held out")
print(f"transport     n_fft={CFG.n_fft} hop={CFG.hop}  high band = bins {CFG.k_cut}..{CFG.n_fft//2}")
print(f"evaluation    n_fft={CFG.eval_n_fft} hop={CFG.eval_hop} (different basis, on purpose)")

## 2. Spectral primitives

Hann STFT with exact inversion, log-magnitude, and the resynthesis helper the log-magnitude ensemble readout
needs. A metric is never computed on a modified spectrogram: we always resynthesise to a waveform and
re-analyse.

In [ ]:
# ---- inlined verbatim from lisa_rtm.ipynb §2 ----
def _hann(n):
    '''Periodic Hann, float64.'''
    return 0.5 - 0.5 * np.cos(2.0 * np.pi * np.arange(n) / n)


def stft(x, n_fft, hop):
    '''Real signal -> (n_frames, n_bins) complex.  Returns (S, pad, length) for exact inversion.'''
    x = np.asarray(x, dtype=np.float64)
    length, pad = len(x), n_fft
    xp = np.pad(x, (pad, pad + n_fft))
    n_frames = 1 + (len(xp) - n_fft) // hop
    idx = np.arange(n_fft)[None, :] + hop * np.arange(n_frames)[:, None]
    return np.fft.rfft(xp[idx] * _hann(n_fft), axis=-1), pad, length


def istft(S, n_fft, hop, pad, length):
    '''Inverse of stft().  Weighted overlap-add; exact on interior samples.'''
    w = _hann(n_fft)
    frames = np.fft.irfft(S, n=n_fft, axis=-1) * w
    n_frames = frames.shape[0]
    total = (n_frames - 1) * hop + n_fft
    out = np.zeros(total)
    wsq = np.zeros(total)
    for i in range(n_frames):
        s = i * hop
        out[s:s + n_fft] += frames[i]
        wsq[s:s + n_fft] += w ** 2
    out /= np.maximum(wsq, 1e-12)
    return out[pad:pad + length]


def logmag(S, eps=1e-8):
    return np.log(np.abs(S) + eps)


def resynth(S_ref, new_logmag, hi_slice, n_fft, hop, pad, length, phase=None, eps=1e-8):
    '''Replace the high band's log-magnitude, keep (or replace) phase, return a waveform.'''
    S = S_ref.copy()
    mag = np.exp(new_logmag) - eps
    mag = np.maximum(mag, 0.0)
    ph = np.angle(S[:, hi_slice]) if phase is None else phase
    S[:, hi_slice] = mag * np.exp(1j * ph)
    return istft(S, n_fft, hop, pad, length)

## 3. Metrics

SNR, LSD, per-third-octave band energy ratio (the direct over-smoothing measure), fair-ensemble CRPS, PIT
ranks and spread–skill.

In [ ]:
# ---- inlined verbatim from lisa_rtm.ipynb §4 ----
def snr_db(y, y_hat):
    y, y_hat = np.asarray(y, float), np.asarray(y_hat, float)
    n = min(len(y), len(y_hat))
    e = y[:n] - y_hat[:n]
    return 10.0 * np.log10(np.sum(y[:n] ** 2) / max(np.sum(e ** 2), 1e-20))


def lsd_db(y, y_hat, n_fft, hop, k_from=0, eps=1e-10):
    '''Log-spectral distance on power spectra (log10), optionally restricted to k >= k_from.'''
    Sy = np.abs(stft(y, n_fft, hop)[0])[:, k_from:]
    Sh = np.abs(stft(y_hat, n_fft, hop)[0])[:, k_from:]
    d = np.log10(Sy ** 2 + eps) - np.log10(Sh ** 2 + eps)
    return float(np.mean(np.sqrt(np.mean(d ** 2, axis=1))))


def third_octave_edges(fs, f_lo, f_hi):
    f = [f_lo]
    while f[-1] < f_hi:
        f.append(f[-1] * 2 ** (1 / 3))
    return np.array(f)


def band_energy_ratio(y, y_hat, fs, n_fft, hop, f_lo, f_hi, frame_mask=None):
    '''rho_b in dB per third-octave band.  0 dB = correct energy, negative = over-smoothed.'''
    Sy, Sh = np.abs(stft(y, n_fft, hop)[0]), np.abs(stft(y_hat, n_fft, hop)[0])
    if frame_mask is not None:
        Sy, Sh = Sy[frame_mask], Sh[frame_mask]
    freqs = np.fft.rfftfreq(n_fft, 1.0 / fs)
    edges = third_octave_edges(fs, f_lo, min(f_hi, fs / 2 - 1))
    out = []
    for a, b in zip(edges[:-1], edges[1:]):
        sel = (freqs >= a) & (freqs < b)
        if sel.sum() == 0:
            continue
        num, den = np.sum(Sh[:, sel] ** 2), np.sum(Sy[:, sel] ** 2)
        out.append((np.sqrt(a * b), 10.0 * np.log10((num + 1e-20) / (den + 1e-20))))
    return np.array(out)          # (n_bands, 2): centre freq, dB


def crps_ensemble(samples, truth):
    '''samples: (M, ...) ensemble;  truth: (...).  Fair estimator, averaged over all elements.'''
    M = samples.shape[0]
    term1 = np.abs(samples - truth[None]).mean(0)
    diff = np.abs(samples[:, None] - samples[None, :]).mean((0, 1))
    return float(np.mean(term1 - 0.5 * diff))


def pit_ranks(samples, truth):
    '''Rank of truth among M samples, in 0..M.  Flat histogram <=> calibrated.'''
    return (samples < truth[None]).sum(0).ravel()


def spread_skill(samples, truth, n_bins=8):
    '''Binned (mean ensemble spread, rmse of ensemble mean).  Unit slope is the target.'''
    spread = samples.std(0).ravel()
    err = (samples.mean(0) - truth).ravel()
    order = np.argsort(spread)
    spread, err = spread[order], err[order]
    edges = np.linspace(0, len(spread), n_bins + 1).astype(int)
    out = []
    for a, b in zip(edges[:-1], edges[1:]):
        if b > a:
            out.append((spread[a:b].mean(), np.sqrt((err[a:b] ** 2).mean())))
    return np.array(out)

## 4. Data utilities

`decimate` is the anti-aliased downsample that makes the high band unrecoverable; `load_utterance` and the
range-request fetcher are kept for the DataShare fixtures the evaluation cells read.

In [ ]:
# ---- inlined verbatim from lisa_rtm.ipynb §6 ----
VCTK_URL = "https://datashare.ed.ac.uk/bitstream/handle/10283/3443/VCTK-Corpus-0.92.zip"


class HTTPRangeFile(io.RawIOBase):
    '''Minimal seekable read-only file over HTTP range requests.'''

    def __init__(self, url, session=None):
        import requests
        self.session = session or requests.Session()
        r = self.session.head(url, allow_redirects=True, timeout=60)
        r.raise_for_status()
        if r.headers.get("Accept-Ranges", "").lower() != "bytes":
            raise OSError("server does not advertise byte ranges")
        self.url, self.length, self.pos = r.url, int(r.headers["Content-Length"]), 0

    def readable(self):  return True
    def seekable(self):  return True
    def tell(self):      return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        self.pos = {io.SEEK_SET: offset,
                    io.SEEK_CUR: self.pos + offset,
                    io.SEEK_END: self.length + offset}[whence]
        return self.pos

    def read(self, size=-1):
        if size < 0:
            size = self.length - self.pos
        size = min(size, self.length - self.pos)
        if size <= 0:
            return b""
        end = self.pos + size - 1
        r = self.session.get(self.url, headers={"Range": f"bytes={self.pos}-{end}"}, timeout=180)
        r.raise_for_status()
        self.pos += len(r.content)
        return r.content


def _members_for(zf, speakers, per_speaker):
    wanted = {}
    for name in zf.namelist():
        if not (name.endswith("_mic1.flac") and "wav48_silence_trimmed/" in name):
            continue
        spk = name.split("/")[-2]
        if spk in speakers:
            wanted.setdefault(spk, []).append(name)
    return {s: sorted(v)[:per_speaker] for s, v in wanted.items()}


def fetch_vctk(speakers, per_speaker, dest):
    '''Extract selected speakers' FLACs into dest/<spk>/.  Returns {speaker: [paths]}.'''
    from tqdm.auto import tqdm
    dest.mkdir(parents=True, exist_ok=True)
    have = {s: sorted((dest / s).glob("*.flac")) for s in speakers}
    if all(len(v) >= per_speaker for v in have.values()):
        return {s: v[:per_speaker] for s, v in have.items()}

    try:
        handle = io.BufferedReader(HTTPRangeFile(VCTK_URL), buffer_size=1 << 20)
        print("opening remote archive over HTTP ranges (no full download)")
    except Exception as e:
        print(f"range requests unavailable ({e}); downloading the full archive once")
        import requests
        local = ROOT / "VCTK-Corpus-0.92.zip"
        if not local.exists():
            with requests.get(VCTK_URL, stream=True, timeout=300) as r:
                r.raise_for_status()
                total = int(r.headers.get("Content-Length", 0))
                with open(local, "wb") as fh, tqdm(total=total, unit="B", unit_scale=True) as bar:
                    for chunk in r.iter_content(1 << 20):
                        fh.write(chunk); bar.update(len(chunk))
        handle = open(local, "rb")

    out = {}
    with zipfile.ZipFile(handle) as zf:
        members = _members_for(zf, set(speakers), per_speaker)
        missing = set(speakers) - set(members)
        if missing:
            raise RuntimeError(f"speakers not found in archive: {sorted(missing)}")
        for spk, names in members.items():
            (dest / spk).mkdir(exist_ok=True)
            paths = []
            for name in tqdm(names, desc=spk, leave=False):
                p = dest / spk / Path(name).name
                if not p.exists():
                    p.write_bytes(zf.read(name))
                paths.append(p)
            out[spk] = paths
    return out


def load_utterance(path, fs_hi, peak=0.95, min_seconds=1.0):
    '''Read a FLAC, mono, resample to fs_hi, peak-normalise.  None if too short.'''
    x, fs = sf.read(str(path), dtype="float64", always_2d=False)
    if x.ndim > 1:
        x = x.mean(1)
    if fs != fs_hi:
        g = math.gcd(int(fs), int(fs_hi))
        x = sps.resample_poly(x, fs_hi // g, fs // g)
    if len(x) < min_seconds * fs_hi:
        return None
    m = np.max(np.abs(x))
    return x * (peak / m) if m > 0 else None


def decimate(x, R):
    '''Anti-aliased downsample by R -- this is what makes the high band unrecoverable.'''
    return sps.resample_poly(x, 1, R)

## 5. Corpus — full VCTK 0.92 (mic1) from the Hub

97 training speakers, 39,639 utterances, 37.3 h at 48 kHz. Held out: p236/p237/p238 (ours) and the paper's own
split (speaker id ≥ 350). The first run downloads ~11 GB of parquet into the runtime and takes a few minutes;
it is cached for the session.

In [ ]:
# ---- packages the Hub loader needs (present on Colab) ----
ensure("huggingface_hub")
ensure("pyarrow")

# ---- inlined verbatim from overnight/cell1_corpus.py ----
# ============================================================ XL-1 corpus
# Full VCTK 0.92 (mic1) from the Hub -- 27 parquet shards, 48 kHz, no DataShare.
# Same loader semantics as load_utterance(): mono, 48 kHz, peak 0.95, drop < 1 s.
import io, time, json, math, numpy as np, soundfile as sf
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import snapshot_download
import pyarrow.parquet as pq

HF_REPO  = "sanchit-gandhi/vctk"
OUR_TEST = {"p236", "p237", "p238"}          # continuity with test_FULL.npz
PAPER_TEST_MIN = 350                          # paper: train = speaker id < 350

t0 = time.time()
local = snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["data/*.parquet"], max_workers=8)
shards = sorted(Path(local).glob("data/*.parquet"))
print(f"{len(shards)} shards downloaded in {time.time()-t0:.0f}s")

def spk_num(s):
    return int(s[1:]) if (s[:1] == "p" and s[1:].isdigit()) else -1

def role(spk):
    if spk in OUR_TEST:
        return "test"
    n = spk_num(spk)
    if n < 0:
        return "skip"
    return "paper_test" if n >= PAPER_TEST_MIN else "train"

def load_bytes(b, fs_hi=CFG.fs_hi, peak=0.95, min_seconds=1.0):
    x, fs = sf.read(io.BytesIO(b), dtype="float64", always_2d=False)
    if x.ndim > 1:
        x = x.mean(1)
    if fs != fs_hi:
        g = math.gcd(int(fs), int(fs_hi))
        x = sps.resample_poly(x, fs_hi // g, fs // g)
    if len(x) < min_seconds * fs_hi:
        return None
    m = float(np.max(np.abs(x)))
    return (x * (peak / m)).astype(np.float32) if m > 0 else None

corpus = {"train": [], "test": [], "paper_test": []}     # (file, spk, array)
pool = ThreadPoolExecutor(32)
n_rows = n_mic1 = 0
t0 = time.time()
for si, sh in enumerate(shards):
    pf = pq.ParquetFile(sh)
    for rg in range(pf.num_row_groups):
        t = pf.read_row_group(rg, columns=["speaker_id", "file", "audio"])
        spks, files, auds = (t.column(c).to_pylist() for c in ("speaker_id", "file", "audio"))
        n_rows += len(files)
        keep = [i for i in range(len(files))
                if ("mic1" in (files[i] or "")) or ("mic1" in ((auds[i] or {}).get("path") or ""))]
        keep = [i for i in keep if role(spks[i]) != "skip"]
        n_mic1 += len(keep)
        arrs = list(pool.map(lambda i: load_bytes(auds[i]["bytes"]), keep))
        for i, a in zip(keep, arrs):
            if a is not None:
                corpus[role(spks[i])].append((files[i], spks[i], a))
        del t, auds
    print(f"  shard {si+1:>2}/{len(shards)}  rows {n_rows:>6}  mic1 kept {n_mic1:>6}  [{time.time()-t0:.0f}s]", flush=True)

for k in corpus:
    corpus[k].sort(key=lambda r: (r[1], r[0]))

# our held-out trio: first 40 utterances per speaker, as in test_FULL.npz
by_spk = {}
for f, s, a in corpus["test"]:
    by_spk.setdefault(s, []).append(a)
test_utts = [a for s in sorted(by_spk) for a in by_spk[s][:40]]
test_spk  = [s for s in sorted(by_spk) for a in by_spk[s][:40]]

train_utts = [a for f, s, a in corpus["train"]]
train_spk  = [s for f, s, a in corpus["train"]]
paper_test_utts = [a for f, s, a in corpus["paper_test"]]
paper_test_spk  = [s for f, s, a in corpus["paper_test"]]

def hours(utts):
    return sum(len(u) for u in utts) / CFG.fs_hi / 3600

print()
print(f"train       {len(train_utts):>6} utts  {len(set(train_spk)):>3} speakers  {hours(train_utts):6.2f} h")
print(f"test (ours) {len(test_utts):>6} utts  {len(set(test_spk)):>3} speakers  {hours(test_utts):6.2f} h   {sorted(set(test_spk))}")
print(f"paper test  {len(paper_test_utts):>6} utts  {len(set(paper_test_spk)):>3} speakers  {hours(paper_test_utts):6.2f} h")
assert not (set(train_spk) & set(test_spk)) and not (set(train_spk) & set(paper_test_spk))
MANIFEST = {"repo": HF_REPO, "train_speakers": sorted(set(train_spk)), "test_speakers": sorted(set(test_spk)),
            "paper_test_speakers": sorted(set(paper_test_spk)), "n_train": len(train_utts), "train_hours": hours(train_utts)}
print("manifest:", json.dumps({k: (v if not isinstance(v, list) else f"{len(v)} items") for k, v in MANIFEST.items()}))

## 6. LISA

The paper's architecture: a four-layer conv encoder (kernels 7,3,3,1; channels 16,32,64,32) whose latents each
see 11 input samples, and a five-layer ReLU MLP of width 144 that reads a relative coordinate plus the three
nearest latents and emits one amplitude at any continuous time coordinate. 86,881 parameters.

In [ ]:
# ---- inlined verbatim from lisa_rtm.ipynb §7 ----
class LISAEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        layers, c_in = [], 1
        for i, (c, k) in enumerate(zip(cfg.enc_channels, cfg.enc_kernels)):
            layers.append(nn.Conv1d(c_in, c, k, padding=k // 2))
            if i < len(cfg.enc_channels) - 1:
                layers.append(nn.ReLU())
            c_in = c
        self.net, self.dim = nn.Sequential(*layers), c_in

    def forward(self, x):                     # (B, L_lo) -> (B, dim, L_lo)
        return self.net(x.unsqueeze(1))


class LISADecoder(nn.Module):
    def __init__(self, latent, cfg):
        super().__init__()
        layers, d = [], 1 + 3 * latent
        for _ in range(cfg.dec_layers - 1):
            layers += [nn.Linear(d, cfg.dec_hidden), nn.ReLU()]
            d = cfg.dec_hidden
        layers.append(nn.Linear(d, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, f):
        return self.net(f).squeeze(-1)


class LISA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.enc = LISAEncoder(cfg)
        self.dec = LISADecoder(self.enc.dim, cfg)
        self.R = cfg.upsample

    def forward(self, x_lo, j0=0, j1=None, perturb=False):
        B, L_lo = x_lo.shape
        j1 = L_lo * self.R if j1 is None else j1
        z = self.enc(x_lo).transpose(1, 2)                      # (B, L_lo, C)
        q = (torch.arange(j0, j1, device=x_lo.device, dtype=torch.float32) / self.R)
        q = q.unsqueeze(0).expand(B, -1)                        # t * fs_lo
        anchor = q + torch.randn_like(q) * 0.5 if perturb else q
        idx = torch.floor(anchor).long().clamp(0, L_lo - 1)
        coord = (2.0 * (q - idx.float()) - 1.0).unsqueeze(-1)

        def take(ii):
            ii = ii.clamp(0, L_lo - 1).unsqueeze(-1).expand(-1, -1, z.shape[-1])
            return torch.gather(z, 1, ii)

        return self.dec(torch.cat([coord, take(idx - 1), take(idx), take(idx + 1)], -1))


class MultiScaleSTFTLoss(nn.Module):
    def __init__(self, n_fft):
        super().__init__()
        self.scales = [(n_fft, n_fft // 4), (n_fft // 2, n_fft // 8), (n_fft // 4, n_fft // 16)]

    def forward(self, y, y_hat):
        total = 0.0
        for n, h in self.scales:
            w = torch.hann_window(n, device=y.device)
            Y = torch.stft(y, n, h, window=w, return_complex=True).abs()
            H = torch.stft(y_hat, n, h, window=w, return_complex=True).abs()
            sc = torch.norm(Y - H, p="fro") / (torch.norm(Y, p="fro") + 1e-8)
            lm = F.l1_loss(torch.log(H + 1e-7), torch.log(Y + 1e-7))
            total = total + sc + lm
        return total / len(self.scales)


model = LISA(CFG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"LISA: {n_params:,} parameters  (paper reports ~89k)")
print(f"  encoder {sum(p.numel() for p in model.enc.parameters()):,}"
      f"   decoder {sum(p.numel() for p in model.dec.parameters()):,}")

## 7. Training helpers

Batch sampling, chunked full-utterance reconstruction, the naive polyphase baseline, and the retrying
checkpoint writer (Colab's Drive mount intermittently loses a directory it wrote to seconds earlier).

In [ ]:
# ---- inlined verbatim from lisa_rtm.ipynb §8 (helpers only) ----
def sample_batch(utts, cfg, rng):
    ys = []
    for _ in range(cfg.batch_size):
        u = utts[rng.integers(len(utts))]
        if len(u) <= cfg.seg_samples:
            u = np.pad(u, (0, cfg.seg_samples - len(u)))
            ys.append(u)
        else:
            s = rng.integers(len(u) - cfg.seg_samples)
            ys.append(u[s:s + cfg.seg_samples])
    y = np.stack(ys).astype(np.float64)
    x = np.stack([decimate(s, cfg.upsample) for s in y])
    return (torch.from_numpy(x).float().to(DEVICE),
            torch.from_numpy(y).float().to(DEVICE))


@torch.no_grad()
def reconstruct(model, y, cfg, chunk=1 << 15):
    '''Full-utterance inference, chunked over query coordinates.'''
    model.eval()
    x_lo = torch.from_numpy(decimate(y, cfg.upsample)).float()[None].to(DEVICE)
    n_out = x_lo.shape[1] * cfg.upsample
    out = [model(x_lo, j0=s, j1=min(s + chunk, n_out)).squeeze(0).cpu().numpy()
           for s in range(0, n_out, chunk)]
    y_hat = np.concatenate(out).astype(np.float64)
    return np.pad(y_hat, (0, max(0, len(y) - len(y_hat))))[:len(y)]


def hb_deficit(model, y, cfg):
    '''Mean high-band energy ratio in dB.  Negative = over-smoothed.'''
    b = band_energy_ratio(y, reconstruct(model, y, cfg), cfg.fs_hi,
                          cfg.eval_n_fft, cfg.eval_hop, cfg.fs_lo / 2, cfg.fs_hi / 2)
    return float(np.mean(b[:, 1])) if len(b) else float("nan")


def naive_upsample(y, cfg):
    '''Polyphase (sinc-windowed) interpolation of the decimated input: the trivial baseline.'''
    y = np.asarray(y, np.float64)
    return sps.resample_poly(decimate(y, cfg.upsample), cfg.upsample, 1)[:len(y)]


def dev_snr(model, y, cfg):
    '''Waveform SNR of the reconstruction and of naive upsampling, both against y.'''
    y = np.asarray(y, np.float64)
    return snr_db(y, reconstruct(model, y, cfg)), snr_db(y, naive_upsample(y, cfg))


def save_ckpt(state, path, retries=4):
    '''Write locally, then copy to `path` with retries.  Colab's Drive mount intermittently claims a
    directory it wrote to seconds ago does not exist (seen at ~20 s write cadence on an A100).  A
    training run must not die of that; if Drive stays unreachable the file is kept in /tmp.'''
    tmp = Path(tempfile.gettempdir()) / f"{path.stem}_{os.getpid()}.pt"
    torch.save(state, tmp)
    err = None
    for attempt in range(retries):
        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(tmp, path)
            return True
        except (OSError, RuntimeError) as e:
            err = e
            time.sleep(3 * (attempt + 1))
    warnings.warn(f"step {state['step']}: checkpoint not written to {path} ({err}); kept at {tmp}")
    return False

## 8. The sampler — `LISAS`, the distances, the energy score

Eight Gaussian channels are concatenated to the input waveform, so only the first convolution changes: +896
weights, 87,777 parameters. With the noise at zero the network is LISA exactly, which is why the deterministic
arms are the same class run at $\tau = 0$ and every arm shares one initialisation.

The two-draw unbiased estimator is
$\tfrac12[d(y,\hat y_1) + d(y,\hat y_2)] - \tfrac12 d(\hat y_1,\hat y_2)$. `HostCorpus` keeps the corpus in host
RAM (a 37 h corpus on the GPU is 32 GB) and moves aligned slices per step. The sequential trainer of this file
is stripped: the stacked trainer below replaces it.

In [ ]:
# ---- inlined verbatim from overnight2/c1_model.py (train_ov2 / time_ov2 stripped) ----
# ============================================================ OV2-1 stochastic LISA + proper scoring rules
# The network becomes the transport map: reference noise (n_noise Gaussian channels at the input
# rate) is pushed through the SAME encoder/decoder as LISA.  eps = 0 recovers LISA exactly, so the
# deterministic arms and the stochastic arms share one class, one parameter count, one init.
#
# Training objectives (all paired on identical batches):
#   det       L1(wave) + lam * MSSTFT(sc + logmag)                 -- the paper's loss (lam = 1e-2)
#   det_split L1(lowpass wave) + lam * MSSTFT                      -- no waveform term above fs_lo/2
#   es_marg   energy score, d = L1(wave) + lam * L1(logmag)        -- proper for per-sample / per-bin marginals
#   es_slice  energy score, d = L1(wave) + lam * sliced-L1(logmag frames)  -- proper for the JOINT law of a frame
#   es_wave   energy score, d = L1(wave)                           -- no spectral term at all
# Energy score with two draws:  ES = (d(y,y1) + d(y,y2))/2 - d(y1,y2)/2  (unbiased, Gneiting & Raftery 2007).
import copy, math, time, json, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import MultiStepLR

N_NOISE = 8


class LISAS(nn.Module):
    def __init__(self, cfg, n_noise=N_NOISE):
        super().__init__()
        self.n_noise, self.R = n_noise, cfg.upsample
        layers, c_in = [], 1 + n_noise
        for i, (c, k) in enumerate(zip(cfg.enc_channels, cfg.enc_kernels)):
            layers.append(nn.Conv1d(c_in, c, k, padding=k // 2))
            if i < len(cfg.enc_channels) - 1:
                layers.append(nn.ReLU())
            c_in = c
        self.enc, self.dim = nn.Sequential(*layers), c_in
        self.dec = LISADecoder(self.dim, cfg)
        self.tau = 0.0          # default noise temperature used by reconstruct() when none is given
        self.seed = 0

    def sample_eps(self, x_lo, tau=1.0, seed=None):
        B, L = x_lo.shape
        if seed is None:
            return tau * torch.randn(B, self.n_noise, L, device=x_lo.device)
        g = torch.Generator(device=x_lo.device).manual_seed(int(seed))
        return tau * torch.randn(B, self.n_noise, L, device=x_lo.device, generator=g)

    def encode(self, x_lo, eps=None):
        B, L = x_lo.shape
        if eps is None:
            eps = torch.zeros(B, self.n_noise, L, device=x_lo.device, dtype=x_lo.dtype)
        return self.enc(torch.cat([x_lo.unsqueeze(1), eps], 1)).transpose(1, 2)      # (B, L, C)

    def decode(self, z, j0=0, j1=None, perturb=False):
        B, L_lo, C = z.shape
        j1 = L_lo * self.R if j1 is None else j1
        q = (torch.arange(j0, j1, device=z.device, dtype=torch.float32) / self.R).unsqueeze(0).expand(B, -1)
        anchor = q + torch.randn_like(q) * 0.5 if perturb else q
        idx = torch.floor(anchor).long().clamp(0, L_lo - 1)
        coord = (2.0 * (q - idx.float()) - 1.0).unsqueeze(-1)

        def take(ii):
            ii = ii.clamp(0, L_lo - 1).unsqueeze(-1).expand(-1, -1, C)
            return torch.gather(z, 1, ii)

        return self.dec(torch.cat([coord, take(idx - 1), take(idx), take(idx + 1)], -1))

    def forward(self, x_lo, j0=0, j1=None, perturb=False, eps=None):
        return self.decode(self.encode(x_lo, eps), j0, j1, perturb)


@torch.no_grad()
def reconstruct(model, y, cfg, chunk=1 << 15, tau=None, seed=None):
    '''Full-utterance inference.  One noise draw per utterance (encoded once), decoded in chunks.
    tau=None -> model.tau (0 for deterministic arms).  Overrides the notebook's reconstruct().'''
    model.eval()
    tau = model.tau if tau is None else tau
    seed = model.seed if seed is None else seed
    x_lo = torch.from_numpy(decimate(np.asarray(y, np.float64), cfg.upsample)).float()[None].to(DEVICE)
    eps = None if tau == 0 else model.sample_eps(x_lo, tau, seed)
    z = model.encode(x_lo, eps)
    n_out = x_lo.shape[1] * cfg.upsample
    out = [model.decode(z, s, min(s + chunk, n_out)).squeeze(0).cpu().numpy() for s in range(0, n_out, chunk)]
    y_hat = np.concatenate(out).astype(np.float64)
    return np.pad(y_hat, (0, max(0, len(y) - len(y_hat))))[:len(y)]


class HostCorpus:
    '''Corpus concatenated in host RAM (float32); batches are aligned slices moved to the GPU per step.
    Same semantics as GPUCorpus, but leaves the whole GPU to the model (a 37 h corpus is 32 GB).'''
    def __init__(self, utts, cfg, seg_hi):
        from concurrent.futures import ThreadPoolExecutor
        R = cfg.upsample
        ys = [np.asarray(u, np.float32) for u in utts if len(u) >= seg_hi]
        ys = [y[: (len(y) // R) * R] for y in ys]
        with ThreadPoolExecutor(32) as ex:
            xs = list(ex.map(lambda y: decimate(y.astype(np.float64), R).astype(np.float32), ys))
        self.n, self.R, self.seg_hi, self.seg_lo = len(ys), R, seg_hi, seg_hi // R
        self.lens = np.array([len(y) for y in ys])
        self.off = np.concatenate([[0], np.cumsum(self.lens)[:-1]])
        self.Y = np.concatenate(ys); del ys
        self.X = np.concatenate(xs); del xs
        self.hours = float(self.lens.sum()) / cfg.fs_hi / 3600
        self._ar_hi = np.arange(seg_hi)
        self._ar_lo = np.arange(seg_hi // R)
        print(f"HostCorpus: {self.n} utts, {self.hours:.2f} h, {(self.Y.nbytes + self.X.nbytes) / 1e9:.1f} GB in host RAM")

    def batch(self, rng, B):
        idx = rng.integers(self.n, size=B)
        n_pos = (self.lens[idx] - self.seg_hi) // self.R + 1
        starts = (rng.random(B) * n_pos).astype(np.int64) * self.R
        hi0 = self.off[idx] + starts
        y = self.Y[hi0[:, None] + self._ar_hi]
        x = self.X[(hi0 // self.R)[:, None] + self._ar_lo]
        return torch.from_numpy(x).to(DEVICE, non_blocking=True), torch.from_numpy(y).to(DEVICE, non_blocking=True)


# ---- distances ------------------------------------------------------------------------------------
SCALES = [(CFG.n_fft, CFG.n_fft // 4), (CFG.n_fft // 2, CFG.n_fft // 8), (CFG.n_fft // 4, CFG.n_fft // 16)]
_WIN = {}

def _win(n, device):
    k = (n, str(device))
    if k not in _WIN:
        _WIN[k] = torch.hann_window(n, device=device)
    return _WIN[k]

def logmag_feats(y):
    '''list over scales of (B, F, T) log-magnitudes.'''
    return [torch.log(torch.stft(y, n, h, window=_win(n, y.device), return_complex=True).abs() + 1e-7)
            for n, h in SCALES]

def lowpass(y, R):
    '''Brick-wall projection onto the input band [0, fs_lo/2] -- a linear projection of the error.'''
    Y = torch.fft.rfft(y)
    k_cut = Y.shape[-1] // R                     # bins <= fs_lo/2
    Y[..., k_cut + 1:] = 0
    return torch.fft.irfft(Y, n=y.shape[-1])

def d_wave(a, b):
    return (a - b).abs().mean(dim=tuple(range(1, a.ndim)))                          # (B,)

def d_logmag(fa, fb):
    return sum((A - B).abs().mean(dim=(1, 2)) for A, B in zip(fa, fb)) / len(fa)       # (B,)

def d_sliced(fa, fb, thetas):
    '''|theta . (A_t - B_t)| averaged over P random unit directions in R^F, frames t, scales.'''
    tot = 0.0
    for A, B, Th in zip(fa, fb, thetas):                       # A: (B, F, T), Th: (F, P)
        D = torch.einsum("bft,fp->bpt", A - B, Th)
        tot = tot + D.abs().mean(dim=(1, 2))
    return tot / len(fa)

def sample_thetas(device):
    out = []
    for n, _ in SCALES:
        Fbins = n // 2 + 1
        th = torch.randn(Fbins, 64, device=device)
        out.append(th / th.norm(dim=0, keepdim=True))
    return out


def arm_loss(kind, lam, m, x, y, spec_loss):
    '''Returns (loss, dict of logged terms).  ES arms run two draws in one 2B forward pass.'''
    if kind in ("det", "det_split"):
        y_hat = m(x, perturb=True)                                            # eps = 0
        l_w = F.l1_loss(lowpass(y_hat, m.R), lowpass(y, m.R)) if kind == "det_split" else F.l1_loss(y_hat, y)
        l_s = spec_loss(y, y_hat)
        return l_w + lam * l_s, {"wave": l_w.item(), "spec": l_s.item()}
    B = x.shape[0]
    eps = m.sample_eps(x.repeat(2, 1), 1.0)
    yh = m(x.repeat(2, 1), perturb=True, eps=eps)
    y1, y2 = yh[:B], yh[B:]
    dw = 0.5 * (d_wave(y, y1) + d_wave(y, y2)) - 0.5 * d_wave(y1, y2)
    if kind == "es_wave":
        loss = dw.mean()
        return loss, {"wave": dw.mean().item(), "spec": 0.0, "spread": d_wave(y1, y2).mean().item()}
    fy, f1, f2 = logmag_feats(y), logmag_feats(y1), logmag_feats(y2)
    if kind == "es_marg":
        d = d_logmag
        ds = 0.5 * (d(fy, f1) + d(fy, f2)) - 0.5 * d(f1, f2)
    elif kind == "es_slice":
        th = sample_thetas(y.device)
        d = lambda a, b: d_sliced(a, b, th)
        ds = 0.5 * (d(fy, f1) + d(fy, f2)) - 0.5 * d(f1, f2)
    else:
        raise ValueError(kind)
    loss = (dw + lam * ds).mean()
    return loss, {"wave": dw.mean().item(), "spec": ds.mean().item(), "spread": d_wave(y1, y2).mean().item()}


def probe_metrics(m, probe, cfg, naive):
    '''SNR / deficit at tau=0 and, for stochastic arms, at tau=1 (one draw).'''
    out = {}
    for tau in ((0.0,) if m.tau == 0 else (0.0, 1.0)):
        yh = reconstruct(m, probe, cfg, tau=tau, seed=0)
        b = band_energy_ratio(probe, yh, cfg.fs_hi, cfg.eval_n_fft, cfg.eval_hop, cfg.fs_lo / 2, cfg.fs_hi / 2)
        out[tau] = (snr_db(probe, yh), float(np.mean(b[:, 1])))
    return out


def load_arm(path, cfg=None):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    cls = globals().get(ck.get("cls", "LISAS"), LISAS)
    m = cls(cfg or CFG, n_noise=ck.get("n_noise", N_NOISE)).to(DEVICE)
    m.load_state_dict(ck["model"]); m.eval()
    m.tau = 0.0 if ck["arm"][0].startswith("det") else 1.0
    return m, ck

print("OV2 model/loss/trainer defined.  LISAS params:", sum(p.numel() for p in LISAS(CFG).parameters()))

## 9. OV3 — decoder-side noise, the joint and ERB geometries, the readout

`LISASD` adds four Gaussian channels per output sample at the decoder input (+576 weights, 88,353 total);
`copy_shared` copies a LISAS into it with the noise columns zeroed, so both classes start identical at
$\varepsilon = 0$. New distances: `d_logmag_l2` (Euclidean on the whole log-magnitude spectrogram, strictly
proper for its joint law), and `d_erb` on 32 triangular bands equally spaced on the ERB-rate scale.
`logmag_ensemble_readout` is the LSD-optimal readout: the per-bin mean of $\log\lvert Y\rvert$ across the draws,
the phase of draw 0, resynthesised, with the given baseband passed through.

In [ ]:
# ---- inlined verbatim from overnight3/e1_model.py ----
# ============================================================ OV3-1 decoder-noise sampler, joint and ERB geometries
# Extends overnight2/c1_model.py (must be exec'd first: LISAS, LISADecoder, arm_loss, d_wave, d_logmag,
# logmag_feats, lowpass, SCALES, _win, N_NOISE).  Four additions, all cheap:
#   LISASD      LISAS plus n_dec Gaussian channels per OUTPUT sample at the decoder input.  The encoder
#               noise alone gives the conditional law only 8 channels at 12 kHz to spread with; decoder
#               noise gives it 4 more per 48 kHz sample (EnScale / DISCO Nets: noise deeper in the net).
#   es_ged      d = L1(wave) + lam * RMS over the whole log-magnitude spectrogram (Euclidean on frames x
#               bins) -- strictly proper for the JOINT law of the spectrogram (Gritsenko et al. 2020),
#               where es_marg's per-bin L1 is proper for the marginals only.
#   es_erb      d = L1(wave) + lam * L1 on log ERB-band magnitudes (32 bands to fs/2 on three STFT scales):
#               the geometry ViSQOL's audio-mode neurogram lives in.  Properness holds for the law of
#               phi(y) whenever d = ||phi(a) - phi(b)||, so this is "score the judge's own space".
#   es_marg_erb d = L1(wave) + lam * (d_logmag + d_erb) / 2.
#   es_split_*  the same with the waveform term on the low-passed band only (det_split for the sampler).
#   logmag_ensemble_readout  mean of log|STFT| over M draws, phase of draw 0, baseband passed through.
import math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F

N_DEC = 4


class LISASD(LISAS):
    '''LISAS with decoder-side noise.  Decoder feature = [coord, z_{i-1}, z_i, z_{i+1}, eps_dec_j], so the
    decoder's first Linear has 1 + 3C + n_dec inputs.  eps is a tuple (eps_enc (B, n_noise, L),
    eps_dec (B, R*L, n_dec)); eps=None -> zeros in both, so tau=0 is LISAS at eps=0 (copy_shared() makes
    the outputs identical).  reconstruct() in c1_model calls encode(x_lo, eps) once and then decode(z, j0, j1)
    in chunks, so encode() stores the full decoder noise on self._eps_dec and decode() slices [j0:j1].'''
    def __init__(self, cfg, n_noise=N_NOISE, n_dec=N_DEC):
        super().__init__(cfg, n_noise)
        self.n_dec = n_dec
        first = self.dec.net[0]
        self.dec.net[0] = nn.Linear(first.in_features + n_dec, first.out_features)
        self._eps_dec = None

    def sample_eps(self, x_lo, tau=1.0, seed=None):
        B, L = x_lo.shape
        g = None if seed is None else torch.Generator(device=x_lo.device).manual_seed(int(seed))
        e = tau * torch.randn(B, self.n_noise, L, device=x_lo.device, generator=g)
        d = tau * torch.randn(B, L * self.R, self.n_dec, device=x_lo.device, generator=g)
        return e, d

    def encode(self, x_lo, eps=None):
        if eps is None:
            e, d = None, None
        elif isinstance(eps, (tuple, list)):
            e, d = eps
        else:
            e, d = eps, None
        self._eps_dec = d
        return super().encode(x_lo, e)

    def decode(self, z, j0=0, j1=None, perturb=False):
        return _decode(self, z, j0, j1, perturb)


# ---- the decoder's first layer, two ways ------------------------------------------------------------
# LISA's decoder feature is [c, z_{i-1}, z_i, z_{i+1}, eps_dec] and the first Linear is linear in those
# blocks, so W1 splits as [w_c | W_z | w_d].  The latent block depends only on the anchor i, so it can be
# evaluated once per INPUT cell -- one length-3 conv1d over the replicate-padded latents at 12 kHz -- instead
# of once per OUTPUT sample at 48 kHz.  Layer-1 MACs per audio second fall from 48,000 x 97 x 144 to
# 12,000 x 96 x 144, and the 97-wide high-rate feature tensor never exists.  This is Shi et al.'s sub-pixel
# convolution: the first layer is a sub-pixel conv whose R phase filters share one bank and differ only by a
# rank-1 bias along w_c.  Anchor jitter is fine: for output j in cell n with phase p the jittered anchor is
# i = n + m and the coordinate is c = 2(q - i) - 1 = c_p - 2m, so the jitter only chooses WHICH row of u is
# read.  Computing c from the gathered i (as below) keeps the arithmetic bit-for-bit the gather path's.
# Both paths stay here so the A/B is one flag, and the RNG draw (randn_like(q)) is identical in both.
LISA_SUBPIXEL = True           # exact; set False for the gather path
LISA_JITTER_PER_CELL = False   # CHANGES MATH: one anchor draw per input cell (see overnight3/e2b_fast.py)


def _decode_gather(self, z, j0=0, j1=None, perturb=False):
    '''The original path: three gathers (one per neighbour) at the output rate, then the full 97/101-wide
    first Linear.  Works for LISAS (n_dec = 0) and LISASD alike.'''
    B, L_lo, C = z.shape
    j1 = L_lo * self.R if j1 is None else j1
    q = (torch.arange(j0, j1, device=z.device, dtype=torch.float32) / self.R).unsqueeze(0).expand(B, -1)
    anchor = q + torch.randn_like(q) * 0.5 if perturb else q
    idx = torch.floor(anchor).long().clamp(0, L_lo - 1)
    coord = (2.0 * (q - idx.float()) - 1.0).unsqueeze(-1)

    def take(ii):
        ii = ii.clamp(0, L_lo - 1).unsqueeze(-1).expand(-1, -1, C)
        return torch.gather(z, 1, ii)

    parts = [coord, take(idx - 1), take(idx), take(idx + 1)]
    if getattr(self, "n_dec", 0):
        d = self._eps_dec
        parts.append(torch.zeros(B, j1 - j0, self.n_dec, device=z.device, dtype=z.dtype)
                     if d is None else d[:, j0:j1].to(z.dtype))
    return self.dec(torch.cat(parts, -1))


def _decode_subpixel(self, z, j0=0, j1=None, perturb=False):
    '''Layer 1 lifted to the input rate; layers 2-5 unchanged.  Exactly _decode_gather, up to the summation
    order of layer 1 (~1e-7 relative in fp32).'''
    B, L_lo, C = z.shape
    R = self.R
    j1 = L_lo * R if j1 is None else j1
    net = self.dec.net
    W1, b1 = net[0].weight, net[0].bias                                  # (H, 1 + 3C + n_dec), (H,)
    H, n_dec = W1.shape[0], getattr(self, "n_dec", 0)
    q = (torch.arange(j0, j1, device=z.device, dtype=torch.float32) / R).unsqueeze(0).expand(B, -1)
    if perturb and LISA_JITTER_PER_CELL:                                 # CHANGES MATH: one draw per cell
        cell = torch.arange(L_lo, device=z.device, dtype=torch.float32)
        a = torch.floor(cell + torch.randn(B, L_lo, device=z.device) * 0.5).clamp(0, L_lo - 1)
        idx = a.long().gather(1, torch.div(torch.arange(j0, j1, device=z.device), R,
                                           rounding_mode="floor").unsqueeze(0).expand(B, -1))
    else:
        anchor = q + torch.randn_like(q) * 0.5 if perturb else q
        idx = torch.floor(anchor).long().clamp(0, L_lo - 1)
    # anchors actually reachable: the whole latent when jittered, only this chunk's cells otherwise
    a0, a1 = (0, L_lo - 1) if perturb else (j0 // R, (j1 - 1) // R)
    zp = torch.cat([z[:, :1], z, z[:, -1:]], 1)                          # replicate pad: (B, L+2, C)
    W_z = W1[:, 1:1 + 3 * C].reshape(H, 3, C).transpose(1, 2)            # taps (A_-1, A_0, A_+1)
    u = F.conv1d(zp[:, a0:a1 + 3].transpose(1, 2), W_z).transpose(1, 2)  # (B, a1-a0+1, H)
    if (not perturb) and j0 % R == 0 and (j1 - j0) % R == 0:
        # gather-free: i = floor(q) = n, so the R outputs of a cell share one row of u and differ only by
        # c_p * w_c.  expand/broadcast backward is a sum-reduction: no atomics, no scatter, no index tensor.
        cp = (2.0 * torch.arange(R, device=z.device, dtype=torch.float32) / R - 1.0).view(1, 1, R, 1)
        h = (u.unsqueeze(2) + cp * W1[:, 0] + b1).reshape(B, j1 - j0, H)
    else:
        coord = (2.0 * (q - idx.float()) - 1.0).unsqueeze(-1)
        h = torch.gather(u, 1, (idx - a0).unsqueeze(-1).expand(B, j1 - j0, H)) + coord * W1[:, 0] + b1
    if n_dec:
        d = self._eps_dec
        d = torch.zeros(B, j1 - j0, n_dec, device=z.device, dtype=z.dtype) if d is None else d[:, j0:j1].to(z.dtype)
        h = h + d @ W1[:, 1 + 3 * C:].t()
    h = F.relu(h) if isinstance(net[1], nn.ReLU) else h
    for mod in net[2:]:
        h = mod(h)
    return h.squeeze(-1)


def _decode(self, z, j0=0, j1=None, perturb=False):
    return (_decode_subpixel if LISA_SUBPIXEL else _decode_gather)(self, z, j0, j1, perturb)


LISAS.decode = _decode          # c1_model's gather path -> the shared switchable one (LISASD.decode delegates)


def copy_shared(src, dst):
    '''Copy every weight of a LISAS into a LISASD (or LISAS); the decoder's extra noise columns are zeroed,
    so dst at eps=0 equals src at eps=0.  Returns dst.'''
    sd, dd = src.state_dict(), dst.state_dict()
    with torch.no_grad():
        for k, v in sd.items():
            if k not in dd:
                continue
            if dd[k].shape == v.shape:
                dd[k].copy_(v)
            elif k.endswith("dec.net.0.weight"):
                dd[k].zero_(); dd[k][:, : v.shape[1]].copy_(v)
    dst.load_state_dict(dd)
    return dst


# ---- distances ------------------------------------------------------------------------------------
def d_logmag_l2(fa, fb):
    '''Frobenius distance on the whole log-magnitude spectrogram divided by sqrt(F*T) (per-element RMS),
    averaged over scales.  Euclidean geometry on frames x bins: strictly proper for the joint law.  (B,)'''
    tot = 0.0
    for A, B in zip(fa, fb):
        tot = tot + (A - B).flatten(1).norm(dim=1) / math.sqrt(A.shape[1] * A.shape[2])
    return tot / len(fa)


def _erb_rate(f):
    '''ERB-rate scale (Glasberg & Moore 1990): 21.4 log10(1 + 4.37 f / 1000).'''
    return 21.4 * np.log10(1.0 + 4.37 * np.asarray(f, np.float64) / 1000.0)


def _erb_rate_inv(e):
    return (10.0 ** (np.asarray(e, np.float64) / 21.4) - 1.0) * 1000.0 / 4.37


_ERB = {}

def erb_filterbank(n_fft, fs, n_bands=32, f_lo=50.0, device=None):
    '''(n_bands, F) triangular filters with centres equally spaced on the ERB-rate scale from f_lo to fs/2.
    Every row sums to 1; every STFT bin at or above f_lo belongs to at least one band (bins no triangle
    reaches, e.g. the exact end points, go to the nearest band by centre frequency; a band too narrow to
    hold a bin takes its nearest bin).  Cached per (n_fft, fs, n_bands, f_lo, device).'''
    key = (int(n_fft), float(fs), int(n_bands), float(f_lo), str(device))
    if key not in _ERB:
        freqs = np.fft.rfftfreq(n_fft, 1.0 / fs)
        edges = _erb_rate_inv(np.linspace(_erb_rate(f_lo), _erb_rate(fs / 2), n_bands + 2))
        W = np.zeros((n_bands, len(freqs)))
        for b in range(n_bands):
            lo, c, hi = edges[b], edges[b + 1], edges[b + 2]
            W[b] = np.clip(np.minimum((freqs - lo) / max(c - lo, 1e-9), (hi - freqs) / max(hi - c, 1e-9)), 0.0, 1.0)
        centres = edges[1:-1]
        for k in np.where((W.sum(0) <= 0) & (freqs >= f_lo))[0]:          # uncovered bins -> nearest band
            W[int(np.argmin(np.abs(centres - freqs[k]))), k] = 1.0
        for b in np.where(W.sum(1) <= 0)[0]:                              # empty bands -> nearest bin
            W[b, int(np.argmin(np.abs(freqs - centres[b])))] = 1.0
        W = W / W.sum(1, keepdims=True)
        _ERB[key] = torch.tensor(W, dtype=torch.float32, device=device)
    return _ERB[key]


def erb_feats(y, fs=None):
    '''list over SCALES of (B, n_bands, T) log ERB-band RMS magnitudes: 0.5 * log(W @ |STFT|^2 + 1e-8).
    The 0.5 puts d_erb on the same scale as d_logmag (log|S|, c1_model), so one lam means one spectral
    weight across es_marg / es_erb and es_marg_erb's average is equal-weight.'''
    fs = CFG.fs_hi if fs is None else fs
    out = []
    for n, h in SCALES:
        P = torch.stft(y, n, h, window=_win(n, y.device), return_complex=True).abs() ** 2
        W = erb_filterbank(n, fs, device=y.device)
        out.append(0.5 * torch.log(torch.matmul(W, P) + 1e-8))
    return out


def d_erb(fa, fb):
    return sum((A - B).abs().mean(dim=(1, 2)) for A, B in zip(fa, fb)) / len(fa)       # (B,)


# ---- arms -------------------------------------------------------------------------------------------
NEW_KINDS = ("es_ged", "es_erb", "es_marg_erb", "es_split_marg", "es_split_marg_erb")
SPLIT_KINDS = ("es_split_marg", "es_split_marg_erb")   # waveform term on the low band only (det_split for the sampler)

def arm_loss3(kind, lam, m, x, y, spec_loss, perturb=True):
    '''c1_model's arm_loss plus the three new geometries.  Same two-draw unbiased estimator
    1/2 [d(y,y1) + d(y,y2)] - 1/2 d(y1,y2), same logged keys (wave, spec, spread).
    Any kind works with LISAS or LISASD: sample_eps / forward handle the class's own noise shape.
    perturb: anchor jitter in the decoder (True in training, as arm_loss hard-codes it; False for a
    validation loss on the same decoder path reconstruct() uses).  es_slice always delegates to arm_loss.'''
    if kind == "es_slice":
        return arm_loss(kind, lam, m, x, y, spec_loss)
    if kind in ("det", "det_split"):
        y_hat = m(x, perturb=perturb)                                          # eps = 0
        l_w = F.l1_loss(lowpass(y_hat, m.R), lowpass(y, m.R)) if kind == "det_split" else F.l1_loss(y_hat, y)
        l_s = spec_loss(y, y_hat)
        return l_w + lam * l_s, {"wave": l_w.item(), "spec": l_s.item()}
    B = x.shape[0]
    eps = m.sample_eps(x.repeat(2, 1), 1.0)
    yh = m(x.repeat(2, 1), perturb=perturb, eps=eps)
    y1, y2 = yh[:B], yh[B:]
    if kind in SPLIT_KINDS:                      # the waveform term never asks for zero above fs_lo/2
        yl, y1l, y2l = lowpass(y, m.R), lowpass(y1, m.R), lowpass(y2, m.R)
        dw = 0.5 * (d_wave(yl, y1l) + d_wave(yl, y2l)) - 0.5 * d_wave(y1l, y2l)
    else:
        dw = 0.5 * (d_wave(y, y1) + d_wave(y, y2)) - 0.5 * d_wave(y1, y2)
    if kind == "es_wave":
        return dw.mean(), {"wave": dw.mean().item(), "spec": 0.0, "spread": d_wave(y1, y2).mean().item()}
    if kind in ("es_marg", "es_split_marg"):
        feats, d = logmag_feats, d_logmag
    elif kind == "es_ged":
        feats, d = logmag_feats, d_logmag_l2
    elif kind == "es_erb":
        feats, d = erb_feats, d_erb
    elif kind in ("es_marg_erb", "es_split_marg_erb"):
        feats = lambda w: (logmag_feats(w), erb_feats(w))
        d = lambda a, b: 0.5 * (d_logmag(a[0], b[0]) + d_erb(a[1], b[1]))
    else:
        raise ValueError(kind)
    fy, f1, f2 = feats(y), feats(y1), feats(y2)
    ds = 0.5 * (d(fy, f1) + d(fy, f2)) - 0.5 * d(f1, f2)
    loss = (dw + lam * ds).mean()
    return loss, {"wave": dw.mean().item(), "spec": ds.mean().item(), "spread": d_wave(y1, y2).mean().item()}


CLASSES = {"LISAS": LISAS, "LISASD": LISASD}

def arm3(spec):
    '''(kind, lam) or (kind, lam, cls_name) -> (kind, lam, cls_name).'''
    return spec[0], spec[1], (spec[2] if len(spec) > 2 else "LISAS")


# ---- ensemble readout ---------------------------------------------------------------------------------
def _split_bands(w, fs, f_cut):
    W = np.fft.rfft(np.asarray(w, np.float64))
    k = int(round(f_cut * len(w) / fs))
    lo, hi = W.copy(), W.copy()
    lo[k:] = 0; hi[:k] = 0
    return np.fft.irfft(lo, n=len(w)), np.fft.irfft(hi, n=len(w))


def logmag_ensemble_readout(draws, y, cfg, passthrough=True):
    '''draws (M, T) waveforms of one utterance -> one waveform of length len(y).  Per-bin MEAN of log|STFT|
    across the draws in the evaluation basis (cfg.eval_n_fft, cfg.eval_hop), phase of draw 0, resynthesised.
    passthrough=True (default): brick-wall low band of naive_upsample(y) + high band of that readout, cut at
    cfg.fs_lo/2 (c7 hybrid).  passthrough=False: the raw readout.'''
    y = np.asarray(y, np.float64)
    n = len(y)
    draws = [np.pad(np.asarray(d, np.float64)[:n], (0, max(0, n - len(d)))) for d in draws]
    S0, pad, length = stft(draws[0], cfg.eval_n_fft, cfg.eval_hop)
    L = logmag(S0)
    for d in draws[1:]:
        L = L + logmag(stft(d, cfg.eval_n_fft, cfg.eval_hop)[0])
    L = L / len(draws)
    w = resynth(S0, L, slice(0, None), cfg.eval_n_fft, cfg.eval_hop, pad, length)
    w = np.pad(w[:n], (0, max(0, n - len(w))))
    if not passthrough:
        return w
    nv = naive_upsample(y, cfg)[:n]; nv = np.pad(nv, (0, n - len(nv)))
    lo, _ = _split_bands(nv, cfg.fs_hi, cfg.fs_lo / 2)
    _, hi = _split_bands(w, cfg.fs_hi, cfg.fs_lo / 2)
    return lo + hi


_p = sum(p.numel() for p in LISASD(CFG).parameters())
print(f"OV3 model/distances defined.  LISASD params: {_p:,}  (LISAS {sum(p.numel() for p in LISAS(CFG).parameters()):,}; "
      f"+{N_DEC} decoder noise channels x {CFG.dec_hidden} = {N_DEC * CFG.dec_hidden} weights)", flush=True)
del _p

## 10. Validation loss and curves

Each arm's **own** objective on a fixed held-out set (speakers p236–p237, utterances disjoint from the
evaluation set), with fixed noise seeds and no anchor jitter, so successive evaluations differ only through
the weights. `plot_curves` redraws train-versus-validation, the loss terms and the probe metrics at every
checkpoint.

In [ ]:
# ---- inlined verbatim from overnight3/e2_trainer.py ----
# ============================================================ OV3-2 paired trainer with validation loss and curves
# train_ov2 (one batch stream, one optimiser per arm) plus: a fixed held-out validation set on which every
# arm's OWN objective is scored every val_every steps (fixed noise seed, no anchor jitter: the decoder path
# reconstruct() uses), an EMA of the training loss at the same steps, the history JSON written at every checkpoint,
# and a curves figure (train/val loss, val terms, probe SNR + deficit) redrawn at every checkpoint.
import copy, math, time, json, numpy as np, torch, matplotlib.pyplot as plt
from torch.optim.lr_scheduler import MultiStepLR


def val_batches(val_corpus, tag, n, batch):
    rng = stream(f"{tag}/val")
    return [val_corpus.batch(rng, batch) for _ in range(n)]


def _fork_devices():
    return [torch.cuda.current_device()] if (torch.cuda.is_available() and DEVICE.type == "cuda") else []


@torch.no_grad()
def val_loss(kind, lam, m, vb, spec_loss, seed=1234):
    '''Mean (loss, wave, spec) of the arm's own objective over the fixed validation batches, eps drawn from a
    fixed seed and perturb=False (no anchor jitter, as in reconstruct), so successive evaluations differ only
    through the weights.'''
    m.eval()
    tot = np.zeros(3)
    with torch.random.fork_rng(devices=_fork_devices()):
        torch.manual_seed(seed)
        for x, y in vb:
            loss, t = arm_loss3(kind, lam, m, x, y, spec_loss, perturb=False)
            tot += (loss.item(), t["wave"], t["spec"])
    m.train()
    return tot / len(vb)


def _nan(v):
    return v is None or (isinstance(v, float) and math.isnan(v))


def _f(vs):
    return [float("nan") if _nan(v) else v for v in vs]


def plot_curves(hist, tag, path=None):
    names = list(hist)
    cols = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    for i, k in enumerate(names):
        h, c = hist[k], cols[i % len(cols)]
        if h["val_step"]:
            ax[0].plot(h["val_step"], h["train_loss_ema"], "-", color=c, lw=1.2, label=f"{k} train (EMA)")
            ax[0].plot(h["val_step"], h["val_loss"], "--", color=c, lw=1.2, label=f"{k} val")
            ax[1].plot(h["val_step"], h["val_wave"], "-", color=c, lw=1.2, label=f"{k} wave")
            ax[1].plot(h["val_step"], h["val_spec"], ":", color=c, lw=1.2, label=f"{k} spec")
        if h["dev_step"]:
            ax[2].plot(h["dev_step"], h["snr0"], "-", color=c, lw=1.2, label=f"{k} SNR tau=0")
            if not all(_nan(v) for v in h["snr1"]):
                ax[2].plot(h["dev_step"], _f(h["snr1"]), "-.", color=c, lw=1.0, label=f"{k} SNR tau=1")
    ax[0].set_yscale("log"); ax[0].set_title("objective: train EMA (solid) vs held-out val (dashed)", fontsize=9)
    ax[0].set_xlabel("step"); ax[0].legend(fontsize=5, ncol=2)
    ax[1].set_yscale("log"); ax[1].set_title("val terms: wave (solid), spec (dotted)", fontsize=9); ax[1].set_xlabel("step"); ax[1].legend(fontsize=5, ncol=2)
    ax[2].set_title("probe utterance: SNR (left), HB deficit dB (right, dashed)", fontsize=9); ax[2].set_xlabel("step"); ax[2].set_ylabel("SNR dB")
    if any(hist[k]["snr_naive"] for k in names):
        ax[2].axhline(hist[names[0]]["snr_naive"][0], color="grey", ls=":", lw=1)
    ax2 = ax[2].twinx()
    for i, k in enumerate(names):
        h, c = hist[k], cols[i % len(cols)]
        if h["dev_step"]:
            d = h["def1"] if not all(_nan(v) for v in h["def1"]) else h["def0"]
            ax2.plot(h["dev_step"], _f(d), "--", color=c, lw=1.0)
    ax2.set_ylabel("deficit dB (one draw for samplers)"); ax2.axhline(0, color="k", lw=0.6)
    ax[2].legend(fontsize=5, ncol=2)
    plt.tight_layout()
    path = path or (FIGS / f"ov3_curves_{tag}.png")
    plt.savefig(path, dpi=130); plt.close(fig)
    return path


def train_ov3(corpus, val_corpus, arms, steps, batch, lr, milestones, gamma, clip, ckpt_every, tag, probe,
              val_every=500, n_val_batches=8, log_every=25):
    '''arms: {name: (kind, lam[, cls_name])}.  One batch stream; one init per class, the LISASD init sharing
    every LISAS weight (copy_shared) so the two classes start identical at eps=0; one optimiser per arm.'''
    names = list(arms)
    specs = {k: arm3(arms[k]) for k in names}
    base = {"LISAS": LISAS(CFG).to(DEVICE)}
    if any(specs[k][2] == "LISASD" for k in names):
        base["LISASD"] = copy_shared(base["LISAS"], LISASD(CFG).to(DEVICE))
    models = {k: copy.deepcopy(base[specs[k][2]]) for k in names}
    for k in names:
        models[k].tau = 0.0 if specs[k][0].startswith("det") else 1.0
    opts = {k: torch.optim.Adam(models[k].parameters(), lr=lr) for k in names}
    scheds = {k: MultiStepLR(opts[k], [int(f * steps) for f in milestones], gamma) for k in names}
    spec_loss = MultiScaleSTFTLoss(CFG.n_fft).to(DEVICE)
    hist = {k: {"step": [], "wave": [], "spec": [], "spread": [], "lr": [],
                "val_step": [], "val_loss": [], "val_wave": [], "val_spec": [], "train_loss_ema": [],
                "dev_step": [], "snr0": [], "def0": [], "snr1": [], "def1": [], "snr_naive": []} for k in names}
    ema = {k: None for k in names}
    vb = val_batches(val_corpus, tag, n_val_batches, batch)
    rng = stream(f"{tag}/batches")
    run_dir = CKPT / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    probe = np.asarray(probe, np.float64)
    naive = snr_db(probe, naive_upsample(probe, CFG))
    logf = open(ROOT / f"train_{tag}.log", "a")
    hist_path = ROOT / f"ov3_history_{tag}.json"
    t0 = time.time()
    for step in range(steps):
        x, y = corpus.batch(rng, batch)
        log = (step % log_every == 0)
        for k in names:
            kind, lam, _ = specs[k]
            m = models[k]
            m.train()
            loss, terms = arm_loss3(kind, lam, m, x, y, spec_loss)
            opts[k].zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), clip)
            opts[k].step()
            scheds[k].step()
            lv = loss.item()
            ema[k] = lv if ema[k] is None else 0.98 * ema[k] + 0.02 * lv
            if log:
                h = hist[k]
                h["step"].append(step); h["wave"].append(terms["wave"]); h["spec"].append(terms["spec"])
                h["spread"].append(terms.get("spread", 0.0)); h["lr"].append(scheds[k].get_last_lr()[0])
        if (step + 1) % val_every == 0 or step + 1 == steps:
            vline = f"  val {step+1:>6}"
            for k in names:
                kind, lam, _ = specs[k]
                vl, vw, vs = val_loss(kind, lam, models[k], vb, spec_loss)
                h = hist[k]
                h["val_step"].append(step + 1); h["val_loss"].append(float(vl)); h["val_wave"].append(float(vw))
                h["val_spec"].append(float(vs)); h["train_loss_ema"].append(float(ema[k]))
                vline += f" | {k}: train {ema[k]:.4f} val {vl:.4f}"
            print(vline, flush=True); logf.write(vline + chr(10)); logf.flush()
        if (step + 1) % ckpt_every == 0 or step + 1 == steps:
            line = f"step {step+1:>6}/{steps} [{time.time()-t0:.0f}s]"
            for k in names:
                kind, lam, cls_name = specs[k]
                m = models[k]
                pm = probe_metrics(m, probe, CFG, naive)
                h = hist[k]
                h["dev_step"].append(step + 1); h["snr_naive"].append(naive)
                h["snr0"].append(pm[0.0][0]); h["def0"].append(pm[0.0][1])
                s1, d1 = pm.get(1.0, (None, None))                   # None (not NaN): valid JSON for det arms
                h["snr1"].append(s1); h["def1"].append(d1)
                save_ckpt({"model": m.state_dict(), "step": step + 1, "history": h, "arm": (kind, lam, cls_name),
                           "n_noise": m.n_noise, "n_dec": getattr(m, "n_dec", 0), "cls": cls_name,
                           "batch": batch, "seg": corpus.seg_hi, "tag": tag}, run_dir / f"{k}.pt")
                line += f" | {k}: w {h['wave'][-1]:.4f} s {h['spec'][-1]:.3f} SNR0 {pm[0.0][0]:5.2f} def0 {pm[0.0][1]:+6.2f}"
                if 1.0 in pm:
                    line += f" SNR1 {s1:5.2f} def1 {d1:+6.2f}"
            line += f"  (naive {naive:.2f})"
            print(line, flush=True)
            logf.write(line + chr(10)); logf.flush()
            try:                                                       # Drive mount can flake; ckpts already saved
                json.dump({k: hist[k] for k in names}, open(hist_path, "w"))
            except OSError as e:
                print("history write failed:", repr(e), flush=True)
            try:
                plot_curves(hist, tag)
            except Exception as e:
                print("plot_curves failed:", repr(e), flush=True)
    logf.close()
    return models, hist


def time_ov3(corpus, arms, batch, n=20):
    names = list(arms)
    specs = {k: arm3(arms[k]) for k in names}
    models = {k: CLASSES[specs[k][2]](CFG).to(DEVICE) for k in names}
    opts = {k: torch.optim.Adam(models[k].parameters(), lr=1e-3) for k in names}
    spec_loss = MultiScaleSTFTLoss(CFG.n_fft).to(DEVICE)
    rng = stream("timing")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    for i in range(n + 3):
        if i == 3:
            sync(); t0 = time.time()
        x, y = corpus.batch(rng, batch)
        for k in names:
            loss, _ = arm_loss3(specs[k][0], specs[k][1], models[k], x, y, spec_loss)
            opts[k].zero_grad(set_to_none=True); loss.backward(); opts[k].step()
    sync()
    dt = (time.time() - t0) / n
    mem = f", peak mem {torch.cuda.max_memory_allocated()/1e9:.1f} GB" if torch.cuda.is_available() else ""
    print(f"{len(names)} arm(s) {names}, batch {batch} x {corpus.seg_hi}: {dt*1000:.0f} ms/step{mem}", flush=True)
    del models, opts
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return dt

print("OV3 trainer defined (train_ov3, time_ov3, val_loss, plot_curves).", flush=True)

## 11. The stacked trainer

Seven arms run sequentially cost ~84 ms per arm-step on an A100 for ~0.7 TFLOP of arithmetic: the step is
launch- and sync-bound, not compute-bound. Here the arms of one class share **one** forward and **one**
backward — parameters carry a leading arm axis, the encoder becomes one grouped convolution per layer and the
decoder one batched matmul per layer. Losses are independent per arm, so a backward on their sum gives each
arm its own gradient; one Adam over the stacked tensors is one Adam per arm; clipping is per arm.

Also: bf16 autocast (losses stay fp32), TF32, a compiled decoder MLP, pinned non-blocking batches, fused Adam,
target STFT features computed once per step and shared by every arm, and no `.item()` between callbacks.

Two things in that file are worth reading before changing them. Its flag block repeats §0a's kernel
selection, because it sits **above** the `torch.compile` and the deterministic flag is mirrored into
Inductor's config at compile time. And decoder layer 1 is evaluated at the 12 kHz **input** rate:
$W_1 = [\,w_c \mid W_z \mid w_d\,]$ is linear in its blocks and the latent block depends only on the anchor,
so $u[i] = A_{-1}z_{i-1} + A_0 z_i + A_{+1} z_{i+1}$ is one grouped `conv1d` over the replicate-padded
latents rather than a $3N$-row gather at 48 kHz. Anchor jitter survives this exactly: the jittered anchor is
$i = n + m$ and the coordinate $c = 2(q-i)-1 = c_p - 2m$, so the jitter only chooses which row of $u$ is
read. With `perturb=False` there is no gather at all — the $R$ outputs of a cell share one row and differ by
$c_p w_c$, which is the periodic shuffle, and its backward is a sum-reduction rather than a scatter.

In [ ]:
# ---- inlined verbatim from overnight3/e2b_fast.py ----
# ============================================================ OV3-2b stacked-arm trainer (GPU-efficient drop-in for e2)
# Same arms, same objectives, same history keys / checkpoint dict / log lines / plot as e2_trainer.py, but the
# arms are trained as STACKS instead of one after another.  Why: 7 arms at batch 32 x 1 s cost 587 ms/step on an
# A100-80GB, ~84 ms per arm-step, the same per-arm cost as batch 64 on 4 Sep -- launch/sync bound (sequential
# arms, ~28 .item() syncs per step, three gathers, fp32), while the arithmetic is ~0.7 TFLOP per arm-step.
#
# What changes (every item behind a flag, read from globals() before this file is exec'd):
#   STACK      arms of one class and one sequence count share ONE forward/backward: parameters carry a leading
#              arm axis, the encoder is one grouped conv per layer (groups = arms), the decoder MLP one batched
#              matmul per layer (baddbmm over the arm axis).  Losses are independent per arm, so backward on
#              their sum gives every arm its own gradient; one Adam over the stacked tensors IS one Adam per arm
#              (elementwise); gradient clipping is per arm (norm along the arm axis).  This is what
#              torch.func.vmap(functional_call) lowers to, written out so compile / streams / graphs see plain ops.
#   AMP        bf16 autocast on encoder + decoder; every distance / STFT in fp32 (outputs cast before the losses).
#   TF32       TF32 matmul/conv (the 4 Sep setting).
#   COMPILE    torch.compile on the decoder MLP call (try/except fallback to eager).
#   PIN        pinned-memory, non_blocking batches (same RNG consumption as HostCorpus.batch -> same batches).
#   FUSED      torch.optim.Adam(fused=True).
#   STREAMS    LISAS and LISASD groups on two CUDA streams; both synced to the default stream before the step.
#              DEFAULT OFF: measured as no-effect-or-worse, and off removes the record_stream/wait_stream
#              bookkeeping in _fwd_bwd.
#   CUDA_GRAPHS EXPERIMENTAL, default False: capture forward+backward+clip of the fixed-shape step into a CUDA
#              graph (optimiser step stays eager).  Untested on the day it was written.
#   GROUP_MAX  chunk a stack into at most this many arms (bounds activation memory).  DEFAULT 1: measured, one
#              arm per stack is both the FASTEST and the smallest row of the table (16.6 / 16.4 audio-s per
#              compute-s at batch 16 / 32, against 13.5 / 13.6 at two per stack and 14.3 / 14.4 at seven), and
#              per-group N = 144 grouped GEMMs tile-quantise badly on an A100, so stacking arms buys nothing.
#   DETERMINISTIC  default False.  See the flag block below: 587 vs 2261 ms/step, measured.
#   SUBPIXEL   default True, EXACT: decoder layer 1 evaluated at the 12 kHz input rate as one grouped conv1d
#              over the latents plus a rank-1 coordinate term (ArmStack._layer1).  Anchor jitter stays per
#              output sample and stays bit-for-bit the gather path's; set False for the A/B.
#   JITTER_PER_CELL  default False, CHANGES MATH: one anchor draw per INPUT cell instead of per output sample.
#              Do not switch this on inside the seven-arm comparison; run it as an eighth paired arm.
#   index gather: base index precomputed once per (L, R); jitter added on the GPU; the three neighbours come from
#              ONE gather on a replicate-padded (A, S, L+2, C) latent (SUBPIXEL=False path only).
#   STFT work: target features once per step, shared by every arm; both draws of every arm in one STFT call.
# Memory (activations saved for backward, decoder dominates): per arm and per output sequence of N = R*L
# samples ~ N * (97 + 4*144) * bytes, i.e. at batch 32 x 1 s (two draws = 64 sequences) ~8.3 GB fp32 / 4.2 GB bf16
# per arm.  A 5-arm LISAS stack at batch 32 bf16 ~21 GB; batch 64 ~42 GB; batch 128 needs GROUP_MAX 2 (~33 GB
# per chunk).  The step is no longer overhead-bound once stacked, so the batch only needs to fill the card.
# NOT gradient checkpointing, deliberately.  Saved activations are ~1.35 kB bf16 per output row (4 hidden x 144
# plus the 97-wide input), which reproduces the measured 8.1 / 15.7 / 30.8 GB at 1 / 2 / 7 arms per stack.
# torch.utils.checkpoint.checkpoint(_mlp_eager, ..., use_reentrant=False, preserve_rng_state=False) would drop
# ~6x of that -- preserve_rng_state=False is safe here only because the MLP block draws no randomness, jitter
# and noise being sampled outside it -- but it costs one extra forward, i.e. +20-40% on a bandwidth-bound step.
# Throughput is FLAT in batch in every measured row, so the headroom buys nothing today, and GROUP_MAX = 1 plus
# SUBPIXEL already cut memory 2-4x.  Revisit ONLY if a re-measured roofline shows the step compute-bound AND a
# larger batch is shown to raise audio-seconds per compute-second.
# Requires in the kernel: c0 boot, c1_model, e1_model, e2_trainer (val_batches, plot_curves, _nan, _f).
import copy, math, time, json, contextlib, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import MultiStepLR

assert "plot_curves" in globals() and "val_batches" in globals(), "exec overnight3/e2_trainer.py before e2b_fast.py"
_CUDA = torch.cuda.is_available() and DEVICE.type == "cuda"
# torch.compile mode.  "max-autotune" turns on Inductor's CUDA-graph trees, whose output tensors live in a
# graph-owned pool that the next replay overwrites; the gradients derived from them are then stale when the
# FUSED Adam reads them, which fails with
#   RuntimeError: accessing tensor output of CUDAGraphs that has been overwritten by a subsequent run
# Measured on this A100 (from the autotune log), ATen already wins the shapes that matter:
#   addmm(3072000x144, 144x144)  bias_addmm 1.73 ms  vs best Triton 1.81 ms
#   mm(144x3072000, 3072000x144) ATen mm    1.34 ms  vs best Triton 18.14 ms  (13x)
# so the sweep buys nothing here and costs minutes of compilation.  Default mode, no CUDA graphs.
FAST = {"STACK": True, "AMP": _CUDA, "TF32": _CUDA, "COMPILE": _CUDA, "PIN": _CUDA, "FUSED": _CUDA,
        "STREAMS": False, "CUDA_GRAPHS": False, "GROUP_MAX": 1, "DETERMINISTIC": False,
        "COMPILE_MODE": "default", "SUBPIXEL": True, "JITTER_PER_CELL": False}
for _k in list(FAST):
    if _k in globals():
        FAST[_k] = globals()[_k]

# ============================================================ kernel selection (READ THIS BEFORE MOVING IT)
# This block must run BEFORE the torch.compile below.  torch.__init__ mirrors use_deterministic_algorithms()
# into torch._inductor.config.deterministic, which switches Inductor's on-device autotuning off, so a compiled
# object built while the flag is on stays on heuristic kernels no matter what the flag does afterwards.
#
# WHY TRAINING RUNS WITH DETERMINISM OFF.  The notebook's boot cell (lisa_rtm.ipynb section 0, seed_everything)
# sets cudnn.deterministic = True, cudnn.benchmark = False and use_deterministic_algorithms(True, warn_only=True).
# Under that flag ATen replaces the fused fastAtomicAdd kernel behind the backward of torch.gather (scatter_add_)
# with _scatter_via_index_put -> index_put_with_sort_kernel, which materialises one int64 key per gathered
# element -- ~147M keys, ~1.2 GB per buffer, per arm-draw at batch 32 -- and radix-sorts them, thirteen
# sequence-passes per step.  It also slows the encoder's conv backward.  The repo's own A/B, same sequential
# trainer, 7 arms, batch 32 x 1 s, A100-80GB:
#
#       flag ON,  cudnn.benchmark off (notebook boot cell)        2261 ms/step
#       flag OFF, cudnn.benchmark on  (overnight/cell2_trainer.py:8-12)   587 ms/step     -> 3.9x
#
# What this does NOT change: seeds, data order, noise draws, jitter draws, initialisation and arm pairing are
# all untouched, so the seven arms still see identical batches and stay comparable.  What it does change: the
# float reduction ORDER inside the gradient atomics, i.e. ~1e-7 relative in fp32, far below the bf16 rounding
# the run already carries.  Run-to-run bitwise reproducibility of the gradients is the price.
# THE EVALUATION PATH KEEPS THE REPO'S REPRODUCIBILITY.  e4_eval / e5_visqol run under no_grad, so the scatter
# backward never runs there and determinism is nearly free; the built notebook re-enables the strict pair in a
# cell above the evaluation sections (build_train_notebook.py), and seed_everything() is unchanged.
# Set DETERMINISTIC = True in a cell above this file to put the strict pair back for training too.
if FAST["DETERMINISTIC"]:
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
else:
    torch.use_deterministic_algorithms(False)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
if FAST["TF32"] and _CUDA:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
_OFF3 = None
_IDX_CACHE = {}
_PHASE_CACHE = {}


def _autocast():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16) if (FAST["AMP"] and _CUDA) else contextlib.nullcontext()


def _base_index(L, R, device):
    '''q = j / R for the full sequence and floor(q), computed once per (L, R, device).'''
    key = (int(L), int(R), str(device))
    if key not in _IDX_CACHE:
        q = torch.arange(L * R, device=device, dtype=torch.float32) / R
        _IDX_CACHE[key] = (q, torch.floor(q).long().clamp(0, L - 1))
    return _IDX_CACHE[key]


def _phase_coord(R, device):
    '''c_p = 2p/R - 1, p = 0..R-1: the coordinate of the R outputs of one input cell when the anchor is
    floor(q), i.e. the jitter-free case.  Cached per (R, device).'''
    key = (int(R), str(device))
    if key not in _PHASE_CACHE:
        _PHASE_CACHE[key] = 2.0 * torch.arange(R, device=device, dtype=torch.float32) / R - 1.0
    return _PHASE_CACHE[key]


def _mlp_eager(X, Ws, bs, relus):
    '''(A, rows, in) -> (A, rows, out).  A == 1 (GROUP_MAX = 1, the default) takes the 2-D F.linear path:
    Inductor's mm templating is stronger than its bmm templating, and the ReLU epilogue is fused into the
    GEMM only when a Triton mm template wins autotuning -- each fused epilogue removes two of the four HBM
    traffic terms of a layer, and the decoder is bandwidth-bound (~72 FLOP/byte at bf16 vs A100's ~153).'''
    if X.shape[0] == 1:
        Y = X[0]
        for W, b, r in zip(Ws, bs, relus):
            Y = F.linear(Y, W[0], b[0])
            if r:
                Y = F.relu(Y)
        return Y.unsqueeze(0)
    for W, b, r in zip(Ws, bs, relus):
        X = torch.baddbmm(b.unsqueeze(1), X, W.transpose(1, 2))
        if r:
            X = F.relu(X)
    return X


_MLP = {"fn": _mlp_eager, "compiled": False}
if FAST["COMPILE"]:
    try:
        # _mlp is called at several shapes per step: A varies per stack, rows = S*N with S = B (det arms),
        # 2B (es arms) and the validation batch, and in_features is 97 (LISAS) or 101 (LISASD).  With
        # dynamic=False every distinct size is a fresh graph, and once the recompile limit is hit Dynamo
        # SKIPS the function and everything nested in it -- the compile silently becomes eager, which is
        # consistent with the stacked runs showing no benefit from COMPILE at all.  Raise the limit and mark
        # the row axis dynamic; the last axis stays static so the template still specialises on width.
        import torch._dynamo
        import torch._inductor.config as _ind_cfg
        _ind_cfg.triton.cudagraphs = False      # fused Adam + cudagraph trees = stale grads
        for _attr in ("recompile_limit", "cache_size_limit"):           # cache_size_limit is now an alias
            if hasattr(torch._dynamo.config, _attr):
                setattr(torch._dynamo.config, _attr, 32)
        _MLP = {"fn": torch.compile(_mlp_eager, dynamic=False, mode=FAST["COMPILE_MODE"]), "compiled": True}
    except Exception as _e:                                            # no compiler: eager
        print("torch.compile unavailable, eager MLP:", repr(_e), flush=True)


def _mlp(X, Ws, bs, relus):
    if _MLP["compiled"]:
        try:
            for _d in (0, 1):                  # never mark a size-1 axis: Dynamo specialises on 0/1 anyway
                if X.shape[_d] > 1:
                    torch._dynamo.mark_dynamic(X, _d)
            return _MLP["fn"](X, Ws, bs, relus)
        except Exception as e:                                          # backend failed at first call: eager
            print("torch.compile failed, falling back to eager MLP:", repr(e)[:200], flush=True)
            _MLP["fn"], _MLP["compiled"] = _mlp_eager, False
    return _mlp_eager(X, Ws, bs, relus)


_OOM = torch.cuda.OutOfMemoryError if _CUDA else MemoryError


def _pkey(k):
    return k.replace(".", "__")


class ArmStack:
    '''A arms of one class (LISAS / LISASD) and one sequence count (det: B, es: 2B), parameters stacked along a
    leading arm axis.  Exact per-arm semantics of arm_loss3: same estimator, same distances, same clipping.'''
    def __init__(self, names, specs, base, cls_name, device, stream=None):
        self.names, self.specs, self.cls_name, self.device = list(names), specs, cls_name, device
        self.A = len(self.names)
        self.kinds = [specs[k][0] for k in self.names]
        self.lams = torch.tensor([float(specs[k][1]) for k in self.names], device=device)
        self.is_det = all(k.startswith("det") for k in self.kinds)
        assert self.is_det or not any(k.startswith("det") for k in self.kinds), "mixed det/es stack"
        self.R, self.C, self.n_noise, self.n_dec = base.R, base.dim, base.n_noise, getattr(base, "n_dec", 0)
        sd = base.state_dict()
        self.keys = list(sd)
        self.P = nn.ParameterDict({_pkey(k): nn.Parameter(v.detach().to(device).unsqueeze(0).repeat(self.A, *([1] * v.dim())).clone())
                                   for k, v in sd.items()})
        mods = list(base.enc)
        self.enc_plan = [(f"enc.{i}.weight", f"enc.{i}.bias", m.kernel_size[0], i + 1 < len(mods) and isinstance(mods[i + 1], nn.ReLU))
                         for i, m in enumerate(mods) if isinstance(m, nn.Conv1d)]
        mods = list(base.dec.net)
        self.dec_plan = [(f"dec.net.{i}.weight", f"dec.net.{i}.bias", i + 1 < len(mods) and isinstance(mods[i + 1], nn.ReLU))
                         for i, m in enumerate(mods) if isinstance(m, nn.Linear)]
        self.relus = tuple(r for _, _, r in self.dec_plan)
        # per-arm loss recipe (weights on the three spectral geometries; low-band waveform term; es_wave)
        w_lm, w_erb, w_l2, split = [], [], [], []
        for kd in self.kinds:
            w_lm.append(1.0 if kd in ("es_marg", "es_split_marg") else 0.5 if kd in ("es_marg_erb", "es_split_marg_erb") else 0.0)
            w_erb.append(1.0 if kd == "es_erb" else 0.5 if kd in ("es_marg_erb", "es_split_marg_erb") else 0.0)
            w_l2.append(1.0 if kd == "es_ged" else 0.0)
            split.append(kd in ("det_split", "es_split_marg", "es_split_marg_erb"))
        t = lambda v: torch.tensor(v, device=device, dtype=torch.float32)
        self.w_lm, self.w_erb, self.w_l2 = t(w_lm).unsqueeze(1), t(w_erb).unsqueeze(1), t(w_l2).unsqueeze(1)
        self.split = torch.tensor(split, device=device).unsqueeze(1)
        self.need_lm, self.need_erb, self.need_l2 = any(w_lm), any(w_erb), any(w_l2)
        self.need_split, self.need_full = any(split), not all(split)
        self.stream = stream
        self.ema = None

    def p(self, k):
        return self.P[_pkey(k)]

    def params(self):
        return list(self.P.values())

    # ---- forward: (S, L) input shared by the arms -> (A, S, N) outputs ------------------------------------
    def forward(self, x_in, eps_enc=None, eps_dec=None, perturb=False, jitter=None):
        A, (S, L) = self.A, x_in.shape
        R, C, N = self.R, self.C, x_in.shape[1] * self.R
        if eps_enc is None:
            eps_enc = x_in.new_zeros(A, S, self.n_noise, L)
        h = torch.cat([x_in.unsqueeze(0).expand(A, S, L).unsqueeze(2), eps_enc], 2)       # (A, S, 1+n, L)
        h = h.transpose(0, 1).reshape(S, A * (1 + self.n_noise), L)
        with _autocast():
            for wk, bk, k, relu in self.enc_plan:
                W, b = self.p(wk), self.p(bk)
                h = F.conv1d(h, W.reshape(-1, W.shape[2], W.shape[3]), b.reshape(-1), padding=k // 2, groups=A)
                if relu:
                    h = F.relu(h)
            z = h.view(S, A, C, L).permute(1, 0, 3, 2)                                     # (A, S, L, C)
            zp = torch.cat([z[:, :, :1], z, z[:, :, -1:]], 2)                                # replicate pad: (A, S, L+2, C)
            q, idx0 = _base_index(L, R, x_in.device)
            Ws = [self.p(w) for w, _, _ in self.dec_plan]
            bs = [self.p(b) for _, b, _ in self.dec_plan]
            if FAST["SUBPIXEL"]:
                X = self._layer1(zp, Ws[0], bs[0], eps_dec, q, S, L, N, perturb, jitter)
                X = _mlp(X, Ws[1:], bs[1:], self.relus[1:])
            else:
                if perturb:
                    jit = torch.randn(A, S, N, device=x_in.device) * 0.5 if jitter is None else jitter
                    idx = torch.floor(q + jit).long().clamp_(0, L - 1)                     # (A, S, N)
                else:
                    idx = idx0.view(1, 1, N).expand(A, S, N)
                coord = (2.0 * (q - idx.float()) - 1.0).unsqueeze(-1)                      # (A, S, N, 1)
                global _OFF3
                if _OFF3 is None or _OFF3.device != x_in.device:
                    _OFF3 = torch.arange(3, device=x_in.device)
                gidx = (idx.unsqueeze(-1) + _OFF3).reshape(A, S, 3 * N)                    # padded rows i-1, i, i+1
                g = torch.gather(zp, 2, gidx.unsqueeze(-1).expand(A, S, 3 * N, C)).reshape(A, S, N, 3 * C)
                parts = [coord.to(g.dtype), g]
                if self.n_dec:
                    parts.append((eps_dec if eps_dec is not None else g.new_zeros(A, S, N, self.n_dec)).to(g.dtype))
                X = torch.cat(parts, -1).reshape(A, S * N, -1)
                X = _mlp(X, Ws, bs, self.relus)
        return X.reshape(A, S, N).float()

    # ---- decoder layer 1 at the INPUT rate (exact; see the module note) --------------------------------
    def _layer1(self, zp, W1, b1, eps_dec, q, S, L, N, perturb, jitter):
        '''W1 = [w_c | W_z | w_d] is linear in its blocks, and the latent block depends only on the anchor i,
        so its contribution u[i] = A_-1 z_{i-1} + A_0 z_i + A_+1 z_{i+1} is ONE grouped conv1d over the
        replicate-padded latents at 12 kHz instead of a 3N-row gather at 48 kHz.  Layer-1 MACs per audio
        second fall from 48,000 x 97 x 144 to 12,000 x 96 x 144, and the 97-wide HR feature tensor and its
        backward leave the graph entirely.  (This is Shi et al.'s sub-pixel convolution: LISA's first layer
        is a sub-pixel conv whose R phase filters share one filter bank and differ only by a rank-1 bias.)

        The anchor jitter does NOT have to be disabled and does NOT break exactness.  For output j in cell
        n = floor(q) with phase p, the jittered anchor is i = clamp(floor(q + eta), 0, L-1) = n + m, and the
        coordinate is c = 2(q - i) - 1 = c_p - 2m.  The jitter only selects WHICH row of u is read; the
        arithmetic is untouched, so computing c from the gathered i keeps this bit-for-bit the gather path.
        What jitter costs is that the u gather stays at the output rate, which is why it is a FLOP win and
        roughly traffic-neutral at HR.  FAST["JITTER_PER_CELL"] (changes-math, default off) is what moves the
        gather down to 12 kHz as well.'''
        A, C, R, dev = self.A, self.C, self.R, zp.device
        H = W1.shape[1]
        # W_z columns are ordered [i-1, i, i+1] x C, so (H, 3, C) -> (H, C, 3) puts the conv taps in the
        # order (A_-1, A_0, A_+1); conv1d's out[o, n] = sum_{c,k} W[o, c, k] * in[c, n+k] then reads
        # zp[n], zp[n+1], zp[n+2] = z[n-1], z[n], z[n+1] under the replicate padding.
        W_z = W1[:, :, 1:1 + 3 * C].reshape(A, H, 3, C).transpose(2, 3).reshape(A * H, C, 3)
        u = F.conv1d(zp.permute(1, 0, 3, 2).reshape(S, A * C, L + 2), W_z, groups=A)        # (S, A*H, L)
        u = u.view(S, A, H, L).permute(1, 0, 3, 2)                                          # (A, S, L, H)
        dt = u.dtype
        w_c, bb = W1[:, :, 0].view(A, 1, 1, H).to(dt), b1.view(A, 1, 1, H).to(dt)
        if perturb and FAST["JITTER_PER_CELL"]:                 # CHANGES MATH: one draw per input cell
            jit = torch.randn(A, S, L, device=dev) * 0.5 if jitter is None else jitter
            cell = torch.arange(L, device=dev, dtype=torch.float32)
            idx = torch.floor(cell + jit).long().clamp_(0, L - 1)                           # (A, S, L)
            m = (idx.float() - cell).unsqueeze(-1).unsqueeze(-1)                            # (A, S, L, 1, 1)
            g = torch.gather(u, 2, idx.unsqueeze(-1).expand(A, S, L, H)).unsqueeze(3)       # (A, S, L, 1, H)
            cp = _phase_coord(R, dev).view(1, 1, 1, R, 1)
            X = (g + (cp - 2.0 * m).to(dt) * w_c.unsqueeze(3) + bb.unsqueeze(3)).reshape(A, S, N, H)
        elif perturb:
            jit = torch.randn(A, S, N, device=dev) * 0.5 if jitter is None else jitter
            idx = torch.floor(q + jit).long().clamp_(0, L - 1)                              # (A, S, N)
            coord = (2.0 * (q - idx.float()) - 1.0).unsqueeze(-1)                           # (A, S, N, 1)
            g = torch.gather(u, 2, idx.unsqueeze(-1).expand(A, S, N, H))
            X = g + coord.to(dt) * w_c + bb
        else:
            # perturb=False: i = floor(q) = n, so there is no gather at all -- the periodic shuffle.  expand's
            # backward is a sum-reduction (no atomics, no index tensor, no scatter), which is why this path is
            # immune to the determinism flag whichever way it is set, and why eval can keep the strict pair.
            cp = _phase_coord(R, dev).view(1, 1, 1, R, 1)
            X = (u.unsqueeze(3) + cp.to(dt) * w_c.unsqueeze(3) + bb.unsqueeze(3)).reshape(A, S, N, H)
        if self.n_dec and eps_dec is not None:      # eps_dec=None is a zero block: exactly a no-op, so skip it
            X = X + torch.matmul(eps_dec.to(dt), W1[:, :, 1 + 3 * C:].transpose(1, 2).unsqueeze(1).to(dt))
        X = X.reshape(A, S * N, H)
        return F.relu(X) if self.relus[0] else X

    def sample_eps(self, S, L, gen=None):
        A, dev = self.A, self.device
        e = torch.randn(A, S, self.n_noise, L, device=dev, generator=gen)
        d = torch.randn(A, S, L * self.R, self.n_dec, device=dev, generator=gen) if self.n_dec else None
        return e, d

    # ---- losses: (loss, wave, spec, spread) each (A,) --------------------------------------------------------
    def losses(self, x, y, tf, eps=None, perturb=True):
        B, T = y.shape
        R = self.R
        if self.is_det:
            yh = self.forward(x, None, None, perturb)                                       # (A, B, T)
            l_w = (yh - y).abs().mean((1, 2))
            if self.need_split:
                l_w = torch.where(self.split[:, 0], (lowpass(yh, R) - tf.y_lo).abs().mean((1, 2)), l_w)
            l_s = 0.0
            for s, (n, h) in enumerate(SCALES):
                H = torch.stft(yh.reshape(self.A * B, T), n, h, window=_win(n, y.device), return_complex=True).abs()
                H = H.view(self.A, B, H.shape[1], H.shape[2])
                Y = tf.mag[s]
                sc = (Y - H).pow(2).sum((1, 2, 3)).sqrt() / (tf.mag_norm[s] + 1e-8)
                lm = (torch.log(H + 1e-7) - tf.lm[s]).abs().mean((1, 2, 3))
                l_s = l_s + sc + lm
            l_s = l_s / len(SCALES)
            loss = l_w + self.lams * l_s
            return loss, l_w, l_s, torch.zeros_like(l_w)
        e, d = self.sample_eps(2 * B, x.shape[1]) if eps is None else eps
        yh = self.forward(x.repeat(2, 1), e, d, perturb)                                    # (A, 2B, T)
        y1, y2 = yh[:, :B], yh[:, B:]
        dwf = lambda a, b: (a - b).abs().mean(-1)                                           # (A, B)
        spread = dwf(y1, y2)
        dw = 0.5 * (dwf(y, y1) + dwf(y, y2)) - 0.5 * spread if self.need_full else None
        if self.need_split:
            y1l, y2l = lowpass(y1, R), lowpass(y2, R)
            dws = 0.5 * (dwf(tf.y_lo, y1l) + dwf(tf.y_lo, y2l)) - 0.5 * dwf(y1l, y2l)
            dw = dws if dw is None else torch.where(self.split, dws, dw)
        spec = torch.zeros_like(dw)
        if self.need_lm or self.need_erb or self.need_l2:
            for s, (n, h) in enumerate(SCALES):
                S_ = torch.stft(yh.reshape(self.A * 2 * B, T), n, h, window=_win(n, y.device), return_complex=True).abs()
                S_ = S_.view(self.A, 2 * B, S_.shape[1], S_.shape[2])
                if self.need_lm or self.need_l2:
                    Lm = torch.log(S_ + 1e-7); L1, L2 = Lm[:, :B], Lm[:, B:]; Ly = tf.lm[s]
                    if self.need_lm:
                        dl = lambda a, b: (a - b).abs().mean((-2, -1))
                        spec = spec + self.w_lm * (0.5 * (dl(Ly, L1) + dl(Ly, L2)) - 0.5 * dl(L1, L2)) / len(SCALES)
                    if self.need_l2:
                        sq = math.sqrt(Lm.shape[2] * Lm.shape[3])
                        d2 = lambda a, b: (a - b).flatten(-2).norm(dim=-1) / sq
                        spec = spec + self.w_l2 * (0.5 * (d2(Ly, L1) + d2(Ly, L2)) - 0.5 * d2(L1, L2)) / len(SCALES)
                if self.need_erb:
                    W = erb_filterbank(n, CFG.fs_hi, device=y.device)
                    E = 0.5 * torch.log(torch.matmul(W, S_.pow(2)) + 1e-8); E1, E2 = E[:, :B], E[:, B:]; Ey = tf.erb[s]
                    de = lambda a, b: (a - b).abs().mean((-2, -1))
                    spec = spec + self.w_erb * (0.5 * (de(Ey, E1) + de(Ey, E2)) - 0.5 * de(E1, E2)) / len(SCALES)
        loss = (dw + self.lams.unsqueeze(1) * spec).mean(1)
        return loss, dw.mean(1), spec.mean(1), spread.mean(1)

    # ---- per-arm gradient clipping (clip_grad_norm_ semantics, one norm per arm) ------------------------------
    def clip_(self, clip):
        grads = [p.grad for p in self.P.values() if p.grad is not None]
        if not grads:
            return
        sq = torch.zeros(self.A, device=self.device)
        for g in grads:
            sq = sq + g.reshape(self.A, -1).pow(2).sum(1)
        coef = (clip / (sq.sqrt() + 1e-6)).clamp(max=1.0)
        for g in grads:
            g.mul_(coef.view(self.A, *([1] * (g.dim() - 1))))

    # ---- export one arm into a plain module (checkpoints, probe, reconstruct) ----------------------------------
    @torch.no_grad()
    def export(self, a, module):
        sd = module.state_dict()
        for k in self.keys:
            sd[k].copy_(self.p(k)[a])
        return module


class TargetFeats:
    '''Per-step features of the target batch y, computed once and shared by every arm of every stack.'''
    def __init__(self, y, R, need_lm, need_erb, need_mag, need_split):
        self.lm, self.erb, self.mag, self.mag_norm = [], [], [], []
        self.y_lo = lowpass(y, R) if need_split else None
        for n, h in SCALES:
            S_ = torch.stft(y, n, h, window=_win(n, y.device), return_complex=True).abs() if (need_lm or need_erb or need_mag) else None
            self.mag.append(S_ if need_mag else None)
            self.mag_norm.append(S_.pow(2).sum().sqrt() if need_mag else None)
            self.lm.append(torch.log(S_ + 1e-7) if (need_lm or need_mag) else None)
            self.erb.append(0.5 * torch.log(torch.matmul(erb_filterbank(n, CFG.fs_hi, device=y.device), S_.pow(2)) + 1e-8) if need_erb else None)


def build_stacks(names, specs, base, device):
    '''Group arms by (class, det/es) in the given order, chunk by GROUP_MAX, assign streams by class.'''
    order, groups = [], {}
    for k in names:
        kind, lam, cls = specs[k]
        groups.setdefault((cls, kind.startswith("det")), []).append(k)
    stacks = []
    streams = {}
    for (cls, det), ks in groups.items():
        if FAST["STREAMS"] and _CUDA:
            streams.setdefault(cls, torch.cuda.Stream())
        gm = FAST["GROUP_MAX"] or len(ks)
        for i in range(0, len(ks), gm):
            stacks.append(ArmStack(ks[i:i + gm], specs, base[cls], cls, device, stream=streams.get(cls)))
    return stacks


def _needs(stacks):
    return (any(s.need_lm for s in stacks), any(s.need_erb for s in stacks), any(s.is_det for s in stacks), any(s.need_split for s in stacks))


def batch_pinned(corpus, rng, B):
    '''HostCorpus.batch with the same RNG consumption (same batches as the sequential trainer), pinned + non_blocking.'''
    if not (FAST["PIN"] and _CUDA):
        return corpus.batch(rng, B)
    idx = rng.integers(corpus.n, size=B)
    n_pos = (corpus.lens[idx] - corpus.seg_hi) // corpus.R + 1
    starts = (rng.random(B) * n_pos).astype(np.int64) * corpus.R
    hi0 = corpus.off[idx] + starts
    y = corpus.Y[hi0[:, None] + corpus._ar_hi]
    x = corpus.X[(hi0 // corpus.R)[:, None] + corpus._ar_lo]
    return (torch.from_numpy(x).pin_memory().to(DEVICE, non_blocking=True),
            torch.from_numpy(y).pin_memory().to(DEVICE, non_blocking=True))


def _fwd_bwd(stacks, x, y, clip):
    '''One step's forward + backward + per-arm clip for every stack; returns the (A,4) term tensors in stack order.
    Streams: each stack runs on its stream after waiting for the default stream; the default stream waits for
    all of them before the optimiser touches the gradients.'''
    need = _needs(stacks)
    tf = TargetFeats(y, stacks[0].R, *need)
    cur = torch.cuda.current_stream() if _CUDA else None
    terms = []
    for s in stacks:
        ctx = torch.cuda.stream(s.stream) if (s.stream is not None) else contextlib.nullcontext()
        if s.stream is not None:
            s.stream.wait_stream(cur)
            for t_ in (x, y):
                t_.record_stream(s.stream)
        with ctx:
            loss, wave, spec, spread = s.losses(x, y, tf, perturb=True)
            loss.sum().backward()
            s.clip_(clip)
            terms.append(torch.stack([loss.detach(), wave.detach(), spec.detach(), spread.detach()], 1))
    if _CUDA:
        for s in stacks:
            if s.stream is not None:
                cur.wait_stream(s.stream)
    return terms


class _GraphedStep:
    '''EXPERIMENTAL (CUDA_GRAPHS=True): capture forward+backward+clip into one CUDA graph on static buffers.
    The optimiser step stays eager.  Gradients are static (never set_to_none between replays).'''
    def __init__(self, stacks, x, y, clip):
        self.stacks, self.clip = stacks, clip
        self.x, self.y = x.clone(), y.clone()
        s = torch.cuda.Stream(); s.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(s):
            for _ in range(3):                                            # warm-up (allocator, autotune)
                for st in stacks:
                    for p in st.params():
                        p.grad = None
                _fwd_bwd(stacks, self.x, self.y, clip)
        torch.cuda.current_stream().wait_stream(s)
        for st in stacks:
            for p in st.params():
                p.grad = None
        self.graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(self.graph):
            self.terms = _fwd_bwd(stacks, self.x, self.y, clip)

    def __call__(self, x, y):
        self.x.copy_(x); self.y.copy_(y)
        self.graph.replay()
        return self.terms


def _make_models(arms):
    names = list(arms)
    specs = {k: arm3(arms[k]) for k in names}
    base = {"LISAS": LISAS(CFG).to(DEVICE)}
    if any(specs[k][2] == "LISASD" for k in names):
        base["LISASD"] = copy_shared(base["LISAS"], LISASD(CFG).to(DEVICE))
    models = {k: copy.deepcopy(base[specs[k][2]]) for k in names}
    for k in names:
        models[k].tau = 0.0 if specs[k][0].startswith("det") else 1.0
    return names, specs, base, models


def _make_opts(stacks, lr, steps, milestones, gamma):
    opts, scheds = [], []
    for s in stacks:
        kw = {"fused": True} if (FAST["FUSED"] and _CUDA) else {}
        opts.append(torch.optim.Adam(s.params(), lr=lr, **kw))
        scheds.append(MultiStepLR(opts[-1], [int(f * steps) for f in milestones], gamma))
    return opts, scheds


@torch.no_grad()
def val_loss_fast(stacks, vb, seed=1234):
    '''Every arm's own objective on the fixed validation batches, stacked, no_grad (+autocast), eps from a fixed
    seed, no anchor jitter.  Returns {arm: (loss, wave, spec)} after ONE host transfer.'''
    need = _needs(stacks)
    acc = [torch.zeros(s.A, 3, device=s.device) for s in stacks]
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    for x, y in vb:
        tf = TargetFeats(y, stacks[0].R, *need)
        for i, s in enumerate(stacks):
            eps = None if s.is_det else s.sample_eps(2 * x.shape[0], x.shape[1], gen)
            loss, wave, spec, _ = s.losses(x, y, tf, eps=eps, perturb=False)
            acc[i] += torch.stack([loss, wave, spec], 1)
    out = {}
    vals = torch.cat(acc, 0).div_(len(vb)).cpu().tolist()
    i = 0
    for s in stacks:
        for k in s.names:
            out[k] = tuple(vals[i]); i += 1
    return out


def gpu_stats():
    '''allocated / reserved / peak / total / free GB and utilisation %, all best-effort.'''
    g = {"alloc_gb": 0.0, "reserved_gb": 0.0, "peak_gb": 0.0, "total_gb": 0.0, "free_gb": 0.0, "util_pct": None}
    if not _CUDA:
        return g
    try:
        g["alloc_gb"] = torch.cuda.memory_allocated() / 1e9
        g["reserved_gb"] = torch.cuda.memory_reserved() / 1e9
        g["peak_gb"] = torch.cuda.max_memory_allocated() / 1e9
        free, total = torch.cuda.mem_get_info()
        g["free_gb"], g["total_gb"] = free / 1e9, total / 1e9
    except Exception:
        pass
    try:
        g["util_pct"] = float(torch.cuda.utilization())
    except Exception:
        pass
    return g


def train_ov3_fast(corpus, val_corpus, arms, steps, batch, lr, milestones, gamma, clip, ckpt_every, tag, probe,
                   val_every=500, n_val_batches=8, log_every=25,
                   on_step=None, on_epoch=None, steps_per_epoch=None):
    '''Drop-in for train_ov3: same signature, history keys, checkpoints, log lines, plot.  One init per class
    (LISASD shares the LISAS weights via copy_shared), one batch stream, arms stacked per class.

    on_step(info) fires every log_every steps and on_epoch(info) whenever steps_per_epoch is crossed; `info`
    carries progress, throughput, GPU memory and the per-arm terms.  The per-arm numbers reach the host in ONE
    .cpu().tolist() at the callback step, so the no-sync design between callbacks is unchanged.'''
    names, specs, base, models = _make_models(arms)
    stacks = build_stacks(names, specs, base, DEVICE)
    # (stack index, index within the stack, index in the concatenated (sum A, 4) terms tensor)
    _off, where = 0, {}
    for si, s in enumerate(stacks):
        for a, k in enumerate(s.names):
            where[k] = (si, a, _off + a)
        _off += s.A
    opts, scheds = _make_opts(stacks, lr, steps, milestones, gamma)
    hist = {k: {"step": [], "wave": [], "spec": [], "spread": [], "lr": [],
                "val_step": [], "val_loss": [], "val_wave": [], "val_spec": [], "train_loss_ema": [],
                "dev_step": [], "snr0": [], "def0": [], "snr1": [], "def1": [], "snr_naive": []} for k in names}
    vb = val_batches(val_corpus, tag, n_val_batches, batch)
    rng = stream(f"{tag}/batches")
    run_dir = CKPT / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    probe = np.asarray(probe, np.float64)
    naive = snr_db(probe, naive_upsample(probe, CFG))
    logf = open(ROOT / f"train_{tag}.log", "a")
    hist_path = ROOT / f"ov3_history_{tag}.json"
    pending = []                                                             # (step, lr, terms tensor) -> host at val/ckpt
    graphed = None
    t0 = time.time()
    # ---- callback state: rolling step time, and the most recent val / probe numbers per arm ----------------
    _VAL_KEYS = ("val_loss", "val_wave", "val_spec")
    _DEV_KEYS = ("snr0", "def0", "snr1", "def1")
    last = {k: {kk: None for kk in _VAL_KEYS + _DEV_KEYS} for k in names}
    dt_hist, t_last, epoch_mark = [], time.time(), 0
    seg_s = corpus.seg_hi / CFG.fs_hi
    spe = float(steps_per_epoch) if steps_per_epoch else None

    def _info(step, terms, lr_):
        '''One host transfer: (sum A, 5) = [ema, loss, wave, spec, spread] per arm.'''
        rows = torch.cat([torch.cat([s.ema.unsqueeze(1) for s in stacks], 0),
                          torch.cat([t.reshape(-1, 4) for t in terms], 0)], 1).cpu().tolist()
        el = time.time() - t0
        ms = 1000 * (sum(dt_hist) / len(dt_hist)) if dt_hist else float("nan")
        sps = 1000.0 / ms if ms and ms == ms and ms > 0 else float("nan")
        done = step + 1
        arms_info = {}
        for k in names:
            r = rows[where[k][2]]
            arms_info[k] = {"loss_ema": r[0], "loss": r[1], "wave": r[2], "spec": r[3], "spread": r[4],
                            **{kk: last[k][kk] for kk in _VAL_KEYS + _DEV_KEYS}}
        return {"tag": tag, "step": done, "steps": steps, "frac": done / steps,
                "epoch": done / spe if spe else None, "epochs_total": steps / spe if spe else None,
                "t_elapsed": el, "ms_per_step": ms, "eta_s": (steps - done) * ms / 1000 if ms == ms else None,
                "samples_per_s": sps * batch, "audio_s_per_s": sps * batch * seg_s,
                "batch": batch, "seg_s": seg_s, "lr": lr_, "gpu": gpu_stats(), "arms": arms_info}

    def flush():
        if not pending:
            return
        T = torch.stack([t for _, _, t in pending]).cpu().tolist()           # one transfer for all pending steps
        for (st, lr_, _), rows in zip(pending, T):
            for k in names:
                r = rows[where[k][2]]
                h = hist[k]
                h["step"].append(st); h["wave"].append(r[1]); h["spec"].append(r[2]); h["spread"].append(r[3]); h["lr"].append(lr_)
        pending.clear()

    for step in range(steps):
        x, y = batch_pinned(corpus, rng, batch)
        for s in stacks:
            for p in s.params():
                if graphed is None:
                    p.grad = None
        if FAST["CUDA_GRAPHS"] and _CUDA:
            if graphed is None:
                graphed = _GraphedStep(stacks, x, y, clip)
            terms = graphed(x, y)
        else:
            terms = _fwd_bwd(stacks, x, y, clip)
        for s, o, sc in zip(stacks, opts, scheds):
            o.step(); sc.step()
        for s, t in zip(stacks, terms):
            s.ema = t[:, 0].clone() if s.ema is None else s.ema * 0.98 + t[:, 0] * 0.02
        _now = time.time(); dt_hist.append(_now - t_last); t_last = _now
        if len(dt_hist) > 50:
            del dt_hist[:-50]
        if step % log_every == 0:
            lr_now = scheds[0].get_last_lr()[0]
            pending.append((step, lr_now, torch.cat([t.reshape(-1, 4) for t in terms], 0)))
            if on_step is not None or (on_epoch is not None and spe and (step + 1) // spe > epoch_mark):
                info = _info(step, terms, lr_now)
                if on_step is not None:
                    try:
                        on_step(info)
                    except Exception as e:
                        print("on_step failed:", repr(e)[:200], flush=True)
                if on_epoch is not None and spe and (step + 1) // spe > epoch_mark:
                    epoch_mark = int((step + 1) // spe)
                    try:
                        on_epoch(info)
                    except Exception as e:
                        print("on_epoch failed:", repr(e)[:200], flush=True)
        if (step + 1) % val_every == 0 or step + 1 == steps:
            vals = val_loss_fast(stacks, vb)
            emas = torch.cat([s.ema for s in stacks]).cpu().tolist()
            vline = f"  val {step+1:>6}"
            i = 0
            for s in stacks:
                for k in s.names:
                    vl, vw, vs = vals[k]; h = hist[k]
                    h["val_step"].append(step + 1); h["val_loss"].append(float(vl)); h["val_wave"].append(float(vw))
                    h["val_spec"].append(float(vs)); h["train_loss_ema"].append(float(emas[i]))
                    last[k].update(val_loss=float(vl), val_wave=float(vw), val_spec=float(vs))
                    vline += f" | {k}: train {emas[i]:.4f} val {vl:.4f}"
                    i += 1
            print(vline, flush=True); logf.write(vline + chr(10)); logf.flush()
        if (step + 1) % ckpt_every == 0 or step + 1 == steps:
            flush()
            line = f"step {step+1:>6}/{steps} [{time.time()-t0:.0f}s]"
            for k in names:
                kind, lam, cls_name = specs[k]
                si, a, _ = where[k]
                m = stacks[si].export(a, models[k])
                pm = probe_metrics(m, probe, CFG, naive)
                h = hist[k]
                h["dev_step"].append(step + 1); h["snr_naive"].append(naive)
                h["snr0"].append(pm[0.0][0]); h["def0"].append(pm[0.0][1])
                s1, d1 = pm.get(1.0, (None, None))
                h["snr1"].append(s1); h["def1"].append(d1)
                last[k].update(snr0=pm[0.0][0], def0=pm[0.0][1], snr1=s1, def1=d1)
                save_ckpt({"model": m.state_dict(), "step": step + 1, "history": h, "arm": (kind, lam, cls_name),
                           "n_noise": m.n_noise, "n_dec": getattr(m, "n_dec", 0), "cls": cls_name,
                           "batch": batch, "seg": corpus.seg_hi, "tag": tag}, run_dir / f"{k}.pt")
                line += f" | {k}: w {h['wave'][-1]:.4f} s {h['spec'][-1]:.3f} SNR0 {pm[0.0][0]:5.2f} def0 {pm[0.0][1]:+6.2f}"
                if 1.0 in pm:
                    line += f" SNR1 {s1:5.2f} def1 {d1:+6.2f}"
            line += f"  (naive {naive:.2f})"
            print(line, flush=True)
            logf.write(line + chr(10)); logf.flush()
            try:
                json.dump({k: hist[k] for k in names}, open(hist_path, "w"))
            except OSError as e:
                print("history write failed:", repr(e), flush=True)
            try:
                plot_curves(hist, tag)
            except Exception as e:
                print("plot_curves failed:", repr(e), flush=True)
    flush()
    if steps and (on_step is not None or on_epoch is not None):      # the log grid rarely lands on the last step
        info = _info(steps - 1, terms, scheds[0].get_last_lr()[0])
        for cb in (on_step, on_epoch if (on_epoch is not None and spe and steps // spe > epoch_mark) else None):
            if cb is not None:
                try:
                    cb(info)
                except Exception as e:
                    print("final callback failed:", repr(e)[:200], flush=True)
    logf.close()
    for k in names:
        si, a, _ = where[k]
        stacks[si].export(a, models[k])
    return models, hist


def time_ov3_fast(corpus, arms, batch, n=20, clip=1e-3):
    names, specs, base, _ = _make_models(arms)
    stacks = build_stacks(names, specs, base, DEVICE)
    opts, _ = _make_opts(stacks, 1e-3, 1000, (0.5,), 0.5)
    rng = stream("timing")
    if _CUDA:
        torch.cuda.reset_peak_memory_stats()
    sync = torch.cuda.synchronize if _CUDA else (lambda: None)
    for i in range(n + 3):
        if i == 3:
            sync(); t0 = time.time()
        x, y = batch_pinned(corpus, rng, batch)
        for s in stacks:
            for p in s.params():
                p.grad = None
        _fwd_bwd(stacks, x, y, clip)
        for o in opts:
            o.step()
    sync()
    dt = (time.time() - t0) / n
    mem = f", peak mem {torch.cuda.max_memory_allocated()/1e9:.1f} GB" if _CUDA else ""
    flags = {k: v for k, v in FAST.items() if v}
    print(f"[fast] {len(names)} arm(s) in {len(stacks)} stack(s), batch {batch} x {corpus.seg_hi}: {dt*1000:.0f} ms/step{mem}  flags {flags}", flush=True)
    del stacks, opts
    if _CUDA:
        torch.cuda.empty_cache()
    return dt


def bench_fast(corpus, arms, batch, group_max=None, streams=None, n=10):
    '''Old (time_ov3, sequential arms) vs new (time_ov3_fast) at one batch size.  OOM is reported, not raised.'''
    keep = dict(FAST)
    if group_max is not None:
        FAST["GROUP_MAX"] = group_max
    if streams is not None:
        FAST["STREAMS"] = streams
    rows = []
    for label, fn in (("sequential (e2 time_ov3)", lambda: time_ov3(corpus, arms, batch, n=n) if "time_ov3" in globals() else float("nan")),
                      ("stacked (e2b time_ov3_fast)", lambda: time_ov3_fast(corpus, arms, batch, n=n))):
        try:
            if _CUDA:
                torch.cuda.reset_peak_memory_stats()
            dt = fn()
            mem = torch.cuda.max_memory_allocated() / 1e9 if _CUDA else float("nan")
            rows.append((label, dt * 1000, mem))
        except torch.cuda.OutOfMemoryError if _CUDA else MemoryError:
            rows.append((label, float("nan"), float("nan")))
            if _CUDA:
                torch.cuda.empty_cache()
    FAST.update(keep)
    print(f"| batch {batch} GROUP_MAX {FAST['GROUP_MAX'] if group_max is None else group_max} | ms/step | peak GB |", flush=True)
    for label, ms, mem in rows:
        print(f"| {label} | {'OOM' if math.isnan(ms) else f'{ms:.0f}'} | {'-' if math.isnan(mem) else f'{mem:.1f}'} |", flush=True)
    return rows


print("OV3 fast trainer defined (train_ov3_fast, time_ov3_fast, bench_fast, val_loss_fast, ArmStack).  flags:",
      {k: v for k, v in FAST.items()}, flush=True)

## 12. Live dashboard

Renders into a single updating output: progress and epoch, throughput (ms/step, steps/s, samples/s, and
seconds of audio per second of compute), **GPU memory** (allocated / reserved / peak / total, free, and
utilisation), a per-arm table (training EMA, validation loss and its terms, probe SNR and high-band deficit at
$\tau = 0$ and $\tau = 1$), and a **per-epoch** table that grows as epochs complete. Updates are throttled to
one every two seconds so the Colab output does not flood.

In [ ]:
# ---- live training dashboard (GPU memory, throughput, per-arm and per-epoch statistics) ----
import time as _time

try:
    from IPython.display import display, update_display, HTML
    _dash_HAS_IPY = True
except Exception:
    _dash_HAS_IPY = False

_dash_MONO = "font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:12px;line-height:1.45"


def _dash_hms(s):
    if s is None or s != s:
        return "--:--:--"
    s = int(max(0.0, s))
    return "%d:%02d:%02d" % (s // 3600, (s // 60) % 60, s % 60)


def _dash_num(v, spec="%.4f", dash="&mdash;"):
    if v is None or (isinstance(v, float) and v != v):
        return dash
    try:
        return spec % v
    except Exception:
        return str(v)


def _dash_bar(frac, n=34):
    frac = 0.0 if (frac is None or frac != frac) else min(max(float(frac), 0.0), 1.0)
    k = int(round(frac * n))
    return "&#9608;" * k + "&#9617;" * (n - k)


def _dash_td(x, style=""):
    return "<td style='padding:1px 7px;text-align:right;" + style + "'>" + str(x) + "</td>"


def _dash_th(x):
    return "<th style='padding:1px 7px;text-align:right;border-bottom:1px solid #8884'>" + str(x) + "</th>"


def make_dashboard(tag, arms, every=2.0):
    '''Returns (on_step, on_epoch) for train_ov3_fast.  Re-renders at most once every `every` seconds.'''
    spec = {k: (arm3(v) if "arm3" in globals() else (v[0], v[1], v[2] if len(v) > 2 else "LISAS")) for k, v in arms.items()}
    names = list(arms)
    dev = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(DEVICE)
    did = "ov3-dash-" + str(tag)
    state = {"last": 0.0, "shown": False, "epochs": [], "info": None}

    def _html(info):
        g = info["gpu"]
        a = info["arms"]
        vals = [a[k].get("val_loss") for k in names if a[k].get("val_loss") is not None]
        best = min(vals) if vals else None
        h = ["<div style='" + _dash_MONO + "'>"]
        h.append("<div style='font-weight:600'>" + str(tag) + " &middot; " + str(len(names)) + " arms &middot; batch "
                 + str(info["batch"]) + " &times; " + _dash_num(info["seg_s"], "%.1f") + " s &middot; " + dev + "</div>")
        ep, ept = info.get("epoch"), info.get("epochs_total")
        h.append("<div>" + _dash_bar(info["frac"]) + "  " + str(info["step"]) + "/" + str(info["steps"])
                 + "  " + _dash_num(100 * info["frac"], "%.1f") + "%"
                 + ("  &middot; epoch " + _dash_num(ep, "%.2f") + " / " + _dash_num(ept, "%.2f") if ep is not None else "")
                 + "  &middot; elapsed " + _dash_hms(info["t_elapsed"]) + "  &middot; ETA " + _dash_hms(info.get("eta_s")) + "</div>")
        h.append("<div>throughput: " + _dash_num(info["ms_per_step"], "%.0f") + " ms/step &middot; "
                 + _dash_num(1000.0 / info["ms_per_step"] if info["ms_per_step"] else float("nan"), "%.2f") + " steps/s &middot; "
                 + _dash_num(info["samples_per_s"], "%.1f") + " samples/s &middot; "
                 + _dash_num(info["audio_s_per_s"], "%.1f") + " audio-s per compute-s &middot; lr "
                 + _dash_num(info["lr"], "%.2e") + "</div>")
        used = (g["alloc_gb"] / g["total_gb"]) if g.get("total_gb") else None
        h.append("<div>GPU mem: " + _dash_num(g["alloc_gb"], "%.1f") + " alloc / " + _dash_num(g["reserved_gb"], "%.1f")
                 + " reserved / " + _dash_num(g["peak_gb"], "%.1f") + " peak / " + _dash_num(g["total_gb"], "%.1f")
                 + " GB total &middot; free " + _dash_num(g["free_gb"], "%.1f") + " GB &middot; util "
                 + _dash_num(g.get("util_pct"), "%.0f") + "%  " + _dash_bar(used, 18) + "</div>")
        h.append("<table style='border-collapse:collapse;margin-top:6px;" + _dash_MONO + "'><tr>"
                 + "".join(_dash_th(c) for c in ("arm", "class", "kind", "lambda", "train EMA", "val loss", "val wave",
                                            "val spec", "SNR t0", "SNR t1", "def t0", "def t1")) + "</tr>")
        for k in names:
            kind, lam, cls = spec[k]
            r = a[k]
            hl = "background:#2e7d3222;font-weight:600" if (best is not None and r.get("val_loss") == best) else ""
            h.append("<tr>" + _dash_td(k, "text-align:left") + _dash_td(cls) + _dash_td(kind) + _dash_td(_dash_num(lam, "%g"))
                     + _dash_td(_dash_num(r.get("loss_ema"))) + _dash_td(_dash_num(r.get("val_loss")), hl) + _dash_td(_dash_num(r.get("val_wave")))
                     + _dash_td(_dash_num(r.get("val_spec"), "%.3f")) + _dash_td(_dash_num(r.get("snr0"), "%.2f"))
                     + _dash_td(_dash_num(r.get("snr1"), "%.2f")) + _dash_td(_dash_num(r.get("def0"), "%+.2f"))
                     + _dash_td(_dash_num(r.get("def1"), "%+.2f")) + "</tr>")
        h.append("</table>")
        if state["epochs"]:
            h.append("<div style='margin-top:6px;font-weight:600'>per epoch</div>")
            h.append("<table style='border-collapse:collapse;" + _dash_MONO + "'><tr>"
                     + "".join(_dash_th(c) for c in ("epoch", "step", "wall")) + "".join(_dash_th(k) for k in names) + "</tr>")
            for row in state["epochs"]:
                h.append("<tr>" + _dash_td(_dash_num(row["epoch"], "%.2f")) + _dash_td(row["step"]) + _dash_td(_dash_hms(row["t"]))
                         + "".join(_dash_td(_dash_num(row["train"][k]) + " / " + _dash_num(row["val"][k])) for k in names) + "</tr>")
            h.append("</table><div style='color:#8888'>cell: train EMA / val loss at the epoch boundary</div>")
        h.append("</div>")
        return "".join(h)

    def _text(info):
        g = info["gpu"]
        return ("[%s] step %d/%d (%.1f%%)  epoch %s  %s ms/step  ETA %s  gpu %.1f/%.1f GB (peak %.1f)"
                % (tag, info["step"], info["steps"], 100 * info["frac"],
                   _dash_num(info.get("epoch"), "%.2f", "-"), _dash_num(info["ms_per_step"], "%.0f", "-"),
                   _dash_hms(info.get("eta_s")), g["alloc_gb"], g["total_gb"], g["peak_gb"]))

    def _render(info, force=False):
        now = _time.time()
        if not force and now - state["last"] < every:
            return
        state["last"] = now
        if not _dash_HAS_IPY:
            print(_text(info), flush=True)
            return
        if state["shown"]:
            update_display(HTML(_html(info)), display_id=did)
        else:
            display(HTML(_html(info)), display_id=did)
            state["shown"] = True

    def on_step(info):
        state["info"] = info
        _render(info, force=info["step"] >= info["steps"])

    def on_epoch(info):
        a = info["arms"]
        state["epochs"].append({"epoch": info.get("epoch"), "step": info["step"], "t": info["t_elapsed"],
                                "train": {k: a[k].get("loss_ema") for k in names},
                                "val": {k: a[k].get("val_loss") for k in names}})
        _render(info, force=True)

    return on_step, on_epoch


print("dashboard ready (make_dashboard); IPython display:", _dash_HAS_IPY)

## 13. Launch

Seven arms, one batch stream, one initialisation per class. `det` and `es_marg` are the 8 September references
at $\lambda = 10^{-2}$; the five new arms sit at $\lambda = 10^{-1}$ and vary one thing each: the spectral
weight alone, the waveform term restricted to the low band, the ERB aggregate term, decoder-side noise, and
both together.

Set `BATCH_OVERRIDE`, `BUDGET_H` or `OV3_TAG` in a cell above to change the recipe.

`GROUP_MAX = 1` — **one arm per stack**. The measured table is not ambiguous: one arm per stack is the only
configuration above 16 audio-seconds per compute-second (16.6 at batch 16, 16.4 at batch 32), while two per
stack (13.5, 13.6) and seven per stack (14.3, 14.4) are *slower* and use two and four times the memory. The
per-group GEMMs are only 144 wide, so they tile-quantise badly on an A100 and stacking arms buys nothing but
activation memory. The old default (two per stack at batch 64) landed at ~61 GB, which is the allocator-thrash
row: 8122 ms/step, 7.9 audio-s per compute-s, half of everything else. `STREAMS = False` for the same reason —
streams measured as no-effect-or-worse, and turning them off drops the `record_stream`/`wait_stream`
bookkeeping. `BATCH` stays at 64 so the batch stream is unchanged.

In [ ]:
# ---- the seven paired arms (overnight3/e3_launch.py) ----
ARMS = {
    "det":             ("det",               1e-2, "LISAS"),    # reference (8 Sep recipe)
    "es_marg":         ("es_marg",           1e-2, "LISAS"),    # reference sampler (8 Sep recipe)
    "es_marg_l0.1":    ("es_marg",           1e-1, "LISAS"),    # spectral term dominant (weight x10)
    "es_split_l0.1":   ("es_split_marg",     1e-1, "LISAS"),    # + waveform term on the low band only
    "es_erb_l0.1":     ("es_marg_erb",       1e-1, "LISAS"),    # + aggregate-proper term on log ERB-band energies
    "es_dec_l0.1":     ("es_marg",           1e-1, "LISASD"),   # noise at the output rate
    "es_dec_erb_l0.1": ("es_marg_erb",       1e-1, "LISASD"),   # both
}
if isinstance(globals().get("ARMS_OVERRIDE"), dict):
    ARMS = dict(ARMS_OVERRIDE)
    print("ARMS_OVERRIDE in effect:", list(ARMS), flush=True)

BATCH = int(globals().get("BATCH_OVERRIDE", 64))
SEG = int(globals().get("SEG_OVERRIDE", 48000))
OV3_TAG = globals().get("OV3_TAG", "OV3_fast")
BUDGET_H = float(globals().get("BUDGET_H", 3.0))
CKPT_EVERY = int(globals().get("CKPT_EVERY", 500))   # checkpoints + history JSON + curves to Drive this often
FAST["GROUP_MAX"] = globals().get("GROUP_MAX", 1)   # measured: 1 arm/stack is the fastest AND the smallest
FAST["STREAMS"] = globals().get("STREAMS", False)   # measured: no effect or worse

import gc, sys, json
for _n in ("models_ov3", "hist_ov3"):
    if _n in globals():
        del globals()[_n]
sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    print("GPU before corpus: %.2f GB allocated, %.2f GB reserved"
          % (torch.cuda.memory_allocated() / 1e9, torch.cuda.memory_reserved() / 1e9), flush=True)

# The Hub loader (cell 5) leaves a dict named `corpus`, so test the TYPE, not the name.
if not isinstance(globals().get("corpus"), HostCorpus):
    corpus = HostCorpus(train_utts, CFG, seg_hi=SEG)
    val_corpus = HostCorpus(test_utts[12:52], CFG, seg_hi=SEG)   # test speakers, utterances disjoint from EVAL12
    _step = max(1, len(train_utts) // 200)          # the 200-utterance fit subset; the full list is 26 GB of RAM
    train_utts, train_spk = train_utts[::_step][:200], train_spk[::_step][:200]
    gc.collect()
else:
    print("reusing corpus / val_corpus from the kernel", flush=True)

steps_per_epoch = corpus.hours * 3600 / (BATCH * SEG / CFG.fs_hi)
dt = time_ov3_fast(corpus, ARMS, BATCH)
steps = max(2000, (int(BUDGET_H * 3600 / dt) // 1000) * 1000)
if isinstance(globals().get("STEPS_OVERRIDE"), int):
    steps = int(STEPS_OVERRIDE)
    print("STEPS_OVERRIDE in effect:", steps, flush=True)
plan = ("plan: %d arms x %d steps = %.1f epochs of %.1f h; est %.2f h at %.0f ms/step; %.0f steps/epoch; "
        "batch %d x %.1f s; val %d utts"
        % (len(ARMS), steps, steps / steps_per_epoch, corpus.hours, steps * dt / 3600, dt * 1000,
           steps_per_epoch, BATCH, SEG / CFG.fs_hi, val_corpus.n))
print(plan, flush=True)
(ROOT / ("train_%s.log" % OV3_TAG)).write_text(plan + chr(10) + json.dumps(ARMS) + chr(10))

on_step, on_epoch = make_dashboard(OV3_TAG, ARMS)
models_ov3, hist_ov3 = train_ov3_fast(corpus, val_corpus, ARMS, steps, BATCH, 1e-3,
                                      (0.2, 0.4, 0.5, 0.6, 0.7, 0.8), 0.5, 1e-3, CKPT_EVERY, OV3_TAG, test_utts[0],
                                      on_step=on_step, on_epoch=on_epoch, steps_per_epoch=steps_per_epoch)
json.dump({k: hist_ov3[k] for k in ARMS}, open(ROOT / ("ov3_history_%s.json" % OV3_TAG), "w"))
print("TRAINING DONE", OV3_TAG, flush=True)

## 14. Evaluation — CRPS, calibration, coherence, and the readouts

Every arm on the held-out Hub set: one draw and the 16-draw ensemble (CRPS, sliced CRPS, PIT, spread–skill,
ensemble-mean metrics, coherent fraction and $\kappa$), the log-magnitude ensemble readouts, and every
condition again with the given baseband passed through.

In [ ]:
OV3_RUN_EVAL = True
OV3_TAG = globals().get("OV3_TAG", "OV3_fast")

# ---- EVALUATION keeps the repo's reproducibility: section 0a turned the strict pair off for TRAINING
# only.  Everything below runs under no_grad (and the decoder's perturb=False path has no scatter at
# all), so determinism is nearly free here.
seed_everything()
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ---- inlined verbatim from overnight3/e4_eval.py ----
# ============================================================ OV3-4 evaluation of every OV3 arm on EVAL12
# overnight2/c3_eval.py's machinery, copied (not exec'd), applied to CKPT / OV3_TAG.  Per arm: one draw and
# the 16-draw ensemble (CRPS, sliced CRPS, PIT, spread-skill, ensemble-mean metrics, coherent fraction and
# kappa), plus the log-magnitude ensemble readouts logmean16 / logmean4 (mean of log|STFT| over the draws,
# phase of draw 0), and every condition again with the baseband passed through (c7 hybrid: low band of the
# naive upsample, high band of the model).  Functions at top level; the main block runs only when
# OV3_RUN_EVAL is unset or True.  Requires: c0 boot, c1_model, e1_model (logmag_ensemble_readout).
import json, time, math, numpy as np, matplotlib.pyplot as plt, torch, soundfile as sf

OV3_TAG = globals().get("OV3_TAG", "OV3_es")
M_DRAWS = int(globals().get("OV3_M", 16))
spread = lambda xs, n: xs[:: max(1, len(xs) // n)][:n]

K_HB = CFG.eval_n_fft // 2 + 1 - CFG.eval_k_cut
_th = stream("ov2/theta").standard_normal((K_HB, 32))       # same directions as c3: sliced CRPS comparable with OV2
THETA = _th / np.linalg.norm(_th, axis=0, keepdims=True)
N_GROUPS = 16
GROUP = np.array_split(np.arange(K_HB), N_GROUPS)

def lm_hb(w):
    return logmag(stft(w, CFG.eval_n_fft, CFG.eval_hop)[0][:, CFG.eval_k_cut:])

def grouped(L):                                   # (T, K_HB) -> (T, N_GROUPS)
    return np.stack([L[:, g].mean(1) for g in GROUP], 1)

_FREQS = np.fft.rfftfreq(CFG.eval_n_fft, 1.0 / CFG.fs_hi)
_EDGES_HB = third_octave_edges(CFG.fs_hi, CFG.fs_lo / 2, CFG.fs_hi / 2 - 1)
HB_SEL = [(_FREQS >= a) & (_FREQS < b) for a, b in zip(_EDGES_HB[:-1], _EDGES_HB[1:]) if ((_FREQS >= a) & (_FREQS < b)).sum()]
HB_CENTRES = [float(np.sqrt(a * b)) for a, b in zip(_EDGES_HB[:-1], _EDGES_HB[1:]) if ((_FREQS >= a) & (_FREQS < b)).sum()]

def coherent(y, w):
    '''Per high-band third-octave: coherent fraction Re<Y,P>/<Y,Y> (= predictable fraction rho for the exact
    conditional mean; unbiased for an ensemble mean) and kappa = Re<Y,P>/<P,P> (1 = informative output energy,
    ~0 = hallucinated).  Plus the baseband pair as an alignment check.'''
    Y = stft(y, CFG.eval_n_fft, CFG.eval_hop)[0]; P = stft(w, CFG.eval_n_fft, CFG.eval_hop)[0]
    n = min(len(Y), len(P)); Y, P = Y[:n], P[:n]
    yy = np.array([np.sum(np.abs(Y[:, s]) ** 2) for s in HB_SEL]); pp = np.array([np.sum(np.abs(P[:, s]) ** 2) for s in HB_SEL])
    yp = np.array([np.sum((Y[:, s] * np.conj(P[:, s])).real) for s in HB_SEL])
    bb = _FREQS < CFG.fs_lo / 2
    bb_yp = np.sum((Y[:, bb] * np.conj(P[:, bb])).real)
    return (yp / np.maximum(yy, 1e-20), yp / np.maximum(pp, 1e-20), float(yy.sum() / max(np.sum(np.abs(Y) ** 2), 1e-20)),
            float(bb_yp / max(np.sum(np.abs(Y[:, bb]) ** 2), 1e-20)))

def wave_metrics(y, w):
    b = band_energy_ratio(y, w, CFG.fs_hi, CFG.eval_n_fft, CFG.eval_hop, 200.0, CFG.fs_hi / 2)
    hb = b[:, 0] >= CFG.fs_lo / 2
    coh, kappa, hb_frac, bb_coh = coherent(y, w)
    return dict(snr=snr_db(y, w), lsd=lsd_db(y, w, CFG.eval_n_fft, CFG.eval_hop),
                hb_lsd=lsd_db(y, w, CFG.eval_n_fft, CFG.eval_hop, CFG.eval_k_cut),
                deficit=float(b[hb, 1].mean()), baseband=float(b[~hb, 1].mean()), curve=b[:, 1].tolist(),
                hb_coh=float(np.mean(coh)), hb_kappa=float(np.mean(kappa)),
                hb_frac=hb_frac, bb_coh=bb_coh, coh=coh.tolist(), kappa=kappa.tolist())

def agg(rows, keys=None):
    keys = keys or [k for k in rows[0] if k not in ("curve", "coh", "kappa")]
    out = {k: float(np.mean([r[k] for r in rows])) for k in keys}
    for arr in ("curve", "coh", "kappa"):
        if arr in rows[0]:
            out[arr] = np.mean([r[arr] for r in rows], 0).tolist()
    out["n"] = len(rows)
    return out

# ---- baseband passthrough (c7 hybrid on a waveform) --------------------------------------------------
def split_bands(w, fs, f_cut):
    W = np.fft.rfft(np.asarray(w, np.float64))
    k = int(round(f_cut * len(w) / fs))
    lo, hi = W.copy(), W.copy()
    lo[k:] = 0; hi[:k] = 0
    return np.fft.irfft(lo, n=len(w)), np.fft.irfft(hi, n=len(w))

def passthrough(y, w):
    '''Low band of the naive upsample of y (the given input) + high band of w, brick-wall at fs_lo/2.'''
    y = np.asarray(y, np.float64); n = len(y)
    nv = naive_upsample(y, CFG)[:n]; nv = np.pad(nv, (0, n - len(nv)))
    w = np.asarray(w, np.float64)[:n]; w = np.pad(w, (0, n - len(w)))
    return split_bands(nv, CFG.fs_hi, CFG.fs_lo / 2)[0] + split_bands(w, CFG.fs_hi, CFG.fs_lo / 2)[1]

def floor_ceiling(y):
    '''passthrough + empty high band; passthrough + true high band.'''
    y = np.asarray(y, np.float64)
    return passthrough(y, np.zeros_like(y)), passthrough(y, y)

def snr_gap_calibrated(M):
    '''Expected SNR(mean of M) - SNR(one draw) for a calibrated sampler: 10 log10(2 / (1 + 1/M)).'''
    return 10 * math.log10(2 / (1 + 1 / M))

# ---- models -----------------------------------------------------------------------------------------
def load_ov3(tag):
    models, arms = {}, {}
    for p in sorted((CKPT / tag).glob("*.pt")):
        models[p.stem], ck = load_arm(p)
        arms[p.stem] = tuple(ck["arm"])
        models[p.stem].eval()
    return models, arms

# ---- one model, M draws per utterance ----------------------------------------------------------------
def ensemble_eval(m, utts, M, tau, label):
    '''Deterministic arms: M=1 and CRPS = MAE of the point forecast.  Stochastic arms additionally get the
    ensemble mean, the logmean readouts (M draws and min(4, M) draws) and passthrough versions of everything.'''
    single, single_pt, mean_rows, mean_pt, crps, scrps, ranks, sk = [], [], [], [], [], [], [], []
    ro_spec = ([(f"logmean{M}", M)] + ([("logmean4", 4)] if M > 4 else [])) if M > 1 else []   # no duplicate keys at M <= 4
    ro_names = [n for n, _ in ro_spec]
    ro = {n: [] for n in ro_names}; ro_pt = {n: [] for n in ro_names}
    truth_frames, pred_frames = [], []
    for ui, y in enumerate(utts):
        y = np.asarray(y, np.float64)
        waves = np.stack([reconstruct(m, y, CFG, tau=tau, seed=s) for s in range(M)])
        truth = lm_hb(y)
        ens = np.stack([lm_hb(w)[:truth.shape[0]] for w in waves])
        crps.append(crps_ensemble(ens, truth))
        scrps.append(crps_ensemble(ens @ THETA, truth @ THETA))
        if M > 1:
            ranks.append(pit_ranks(ens, truth)); sk.append(spread_skill(ens, truth))
            wm = waves.mean(0)
            mean_rows.append(wave_metrics(y, wm)); mean_pt.append(wave_metrics(y, passthrough(y, wm)))
            for n, k in ro_spec:
                w_r = logmag_ensemble_readout(waves[:k], y, CFG, passthrough=False)
                ro[n].append(wave_metrics(y, w_r)); ro_pt[n].append(wave_metrics(y, passthrough(y, w_r)))
        single.append(wave_metrics(y, waves[0])); single_pt.append(wave_metrics(y, passthrough(y, waves[0])))
        truth_frames.append(grouped(truth)); pred_frames.append(grouped(ens[0]))
    Tt, Tp = np.vstack(truth_frames), np.vstack(pred_frames)
    corr_err = float(np.linalg.norm(np.corrcoef(Tt.T) - np.corrcoef(Tp.T)))
    res = {"single": agg(single), "single_pt": agg(single_pt), "crps": float(np.mean(crps)), "sliced_crps": float(np.mean(scrps)),
           "corr_err": corr_err, "tau": tau, "M": M}
    if M > 1:
        res["mean"] = agg(mean_rows); res["mean_pt"] = agg(mean_pt)
        res["readouts"] = {n: agg(ro[n]) for n in ro_names}
        res["readouts_pt"] = {n: agg(ro_pt[n]) for n in ro_names}
        r = np.concatenate(ranks)
        h = np.histogram(r, bins=M + 1, range=(-0.5, M + 0.5))[0]
        res["pit_hist"] = h.tolist()
        res["pit_end"] = float((h[0] + h[-1]) / max(h.sum(), 1))          # ideal 2/(M+1)
        res["spread_skill"] = np.mean(sk, axis=0).tolist()
        res["snr_gap"] = res["mean"]["snr"] - res["single"]["snr"]
        res["snr_gap_calibrated"] = snr_gap_calibrated(M)
    s = res["single"]
    line = (f"{label:<26} SNR {s['snr']:6.2f} LSD {s['lsd']:.3f} HB-LSD {s['hb_lsd']:.3f} def {s['deficit']:+6.2f} "
            f"| CRPS {res['crps']:.4f} sCRPS {res['sliced_crps']:.4f} | HB kappa {s['hb_kappa']:+.3f} | pt LSD {res['single_pt']['lsd']:.3f}")
    if M > 1:
        mm = res["mean"]; lm = res["readouts_pt"][ro_names[0]]
        line += (f" | mean: SNR {mm['snr']:6.2f} def {mm['deficit']:+6.2f} gap {res['snr_gap']:.2f} (cal {res['snr_gap_calibrated']:.2f}) "
                 f"PIT end {res['pit_end']:.3f} | {ro_names[0]}+pt LSD {lm['lsd']:.3f} def {lm['deficit']:+6.2f}")
    print(line, flush=True)
    return res

# ---- table / figures / audio ------------------------------------------------------------------------
def make_table(RES, M):
    ro = f"logmean{M}"
    lines = [f"| arm | SNR | LSD | HB-LSD | deficit dB | CRPS | sliced CRPS | HB kappa | PIT end-bins | mean-of-{M} SNR | mean deficit | SNR gap (calibrated {snr_gap_calibrated(M):.2f}) | one-draw passthrough LSD | {ro} LSD |",
             "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for k, r0 in RES.items():
        if k.startswith("_"):
            continue
        r = r0["eval12"]; s = r["single"]
        row = (f"| {k} | {s['snr']:.2f} | {s['lsd']:.3f} | {s['hb_lsd']:.3f} | {s['deficit']:+.2f} | {r['crps']:.4f} | {r['sliced_crps']:.4f} | "
               f"{s['hb_kappa']:+.3f} | ")
        if "mean" in r:
            mm = r["mean"]
            row += (f"{r['pit_end']:.3f} | {mm['snr']:.2f} | {mm['deficit']:+.2f} | {r['snr_gap']:.2f} | {r['single_pt']['lsd']:.3f} | "
                    f"{r['readouts_pt'][ro]['lsd']:.3f} |")
        else:
            row += f"- | - | - | - | {r['single_pt']['lsd']:.3f} | - |"
        lines.append(row)
    lines.append(f"| floor: passthrough + empty HB | {RES['_floor']['snr']:.2f} | {RES['_floor']['lsd']:.3f} | {RES['_floor']['hb_lsd']:.3f} | {RES['_floor']['deficit']:+.2f} | - | - | - | - | - | - | - | - | - |")
    lines.append(f"| ceiling: passthrough + true HB | {RES['_ceiling']['snr']:.2f} | {RES['_ceiling']['lsd']:.3f} | {RES['_ceiling']['hb_lsd']:.3f} | {RES['_ceiling']['deficit']:+.2f} | - | - | - | - | - | - | - | - | - |")
    lines.append(f"\nEVAL12 n={RES['_n']}; PIT end-bins ideal {2/(M+1):.3f}; one-draw passthrough LSD is draw 0 (seed 0) with the baseband passed "
                 f"through; {ro} LSD is the {M}-draw log-magnitude readout with baseband passthrough (raw readout in the JSON). "
                 f"CRPS uses the notebook's crps_ensemble: all-pairs spread term over M^2 pairs incl. the diagonal (standard, not the fair "
                 f"M(M-1) estimator; spread under-credited by (M-1)/M), the same estimator as the OV2 tables.")
    return chr(10).join(lines)


def make_figures(RES, models, utts, tag, M):
    ro = f"logmean{M}"
    centres = band_energy_ratio(utts[0], utts[0], CFG.fs_hi, CFG.eval_n_fft, CFG.eval_hop, 200.0, CFG.fs_hi / 2)[:, 0]
    arms = [k for k in RES if not k.startswith("_")]
    fig, ax = plt.subplots(figsize=(8.5, 4.6))
    cols = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    for i, k in enumerate(arms):
        r = RES[k]["eval12"]; c = cols[i % len(cols)]
        ax.semilogx(centres, r["single"]["curve"], "o-", ms=3, color=c, label=f"{k} (one draw)" if "mean" in r else k)
        if "mean" in r:
            ax.semilogx(centres, r["mean"]["curve"], "--", lw=1, color=c, label=f"{k} (mean of {M})")
            ax.semilogx(centres, r["readouts"][ro]["curve"], ":", lw=1.2, color=c, label=f"{k} ({ro})")
    ax.axhline(0, color="k", lw=1); ax.axvline(CFG.fs_lo / 2, color="crimson", ls="--"); ax.axhspan(-2, 2, color="grey", alpha=0.15)
    ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("pred/target energy (dB)")
    ax.set_title(f"Energy ratio per band: one draw (solid), ensemble mean (dashed), {ro} readout (dotted)", fontsize=9)
    ax.legend(fontsize=6, ncol=2)
    plt.tight_layout(); plt.savefig(FIGS / f"ov3_spectrum_{tag}.png", dpi=130); plt.close(fig)

    stoch = [k for k in arms if "mean" in RES[k]["eval12"]]
    if stoch:
        fig, ax = plt.subplots(1, len(stoch) + 1, figsize=(3.4 * (len(stoch) + 1), 3.2))
        ax = np.atleast_1d(ax)
        for a, k in zip(ax, stoch):
            h = np.array(RES[k]["eval12"]["pit_hist"], float); h /= h.sum()
            a.bar(range(len(h)), h, edgecolor="k", lw=0.5); a.axhline(1 / len(h), color="crimson", ls="--")
            a.set_title(f"PIT {k}", fontsize=9)
        for k in stoch:
            s = np.array(RES[k]["eval12"]["spread_skill"])
            ax[-1].plot(s[:, 0], s[:, 1], "o-", ms=3, label=k)
        lim = ax[-1].get_xlim(); ax[-1].plot([0, lim[1]], [0, lim[1]], "k--", lw=1); ax[-1].set_title("spread-skill", fontsize=9); ax[-1].legend(fontsize=6)
        plt.tight_layout(); plt.savefig(FIGS / f"ov3_calibration_{tag}.png", dpi=130); plt.close(fig)


def write_audio(models, utts, out_dir, M, idx=(0, 5, 10)):
    out_dir.mkdir(parents=True, exist_ok=True)
    def w(name, x):
        sf.write(out_dir / f"{name}.wav", np.clip(np.asarray(x, np.float64), -1, 1), CFG.fs_hi)
    for ui in idx:
        if ui >= len(utts):
            continue
        y = np.asarray(utts[ui], np.float64)
        fl, ce = floor_ceiling(y)
        w(f"u{ui}_truth", y); w(f"u{ui}_naive", naive_upsample(y, CFG)); w(f"u{ui}_floor_pt", fl); w(f"u{ui}_ceiling_pt", ce)
        for k, m in models.items():
            d0 = reconstruct(m, y, CFG, tau=m.tau, seed=0)
            w(f"u{ui}_{k}", d0); w(f"u{ui}_{k}_pt", passthrough(y, d0))
            if m.tau > 0:
                d1 = reconstruct(m, y, CFG, tau=1.0, seed=1)
                w(f"u{ui}_{k}_draw2", d1); w(f"u{ui}_{k}_draw2_pt", passthrough(y, d1))
                draws = np.stack([d0, d1] + [reconstruct(m, y, CFG, tau=1.0, seed=s) for s in range(2, M)])
                w(f"u{ui}_{k}_logmean{M}", logmag_ensemble_readout(draws, y, CFG, passthrough=False))
                w(f"u{ui}_{k}_logmean{M}_pt", logmag_ensemble_readout(draws, y, CFG, passthrough=True))
    print("audio written to", out_dir, flush=True)


def run_eval(models, arms, utts, M, tag, audio=True):
    out = ROOT / "ov3"; out.mkdir(parents=True, exist_ok=True)
    utts = [np.asarray(u, np.float64) for u in utts]
    RES, t0 = {}, time.time()
    for k, m in models.items():
        stoch = m.tau > 0
        RES[k] = {"arm": list(arms[k]), "eval12": ensemble_eval(m, utts, M if stoch else 1, 1.0 if stoch else 0.0, f"{k} tau={m.tau:g}")}
        print(f"  [{time.time()-t0:.0f}s]", flush=True)
    fl, ce = zip(*[floor_ceiling(y) for y in utts])
    RES["_floor"] = agg([wave_metrics(y, f) for y, f in zip(utts, fl)])
    RES["_ceiling"] = agg([wave_metrics(y, c) for y, c in zip(utts, ce)])
    RES["_n"] = len(utts); RES["_M"] = M
    json.dump(RES, open(out / f"results_{tag}.json", "w"), indent=1)
    TABLE = make_table(RES, M)
    (out / f"table_{tag}.md").write_text(TABLE)
    make_figures(RES, models, utts, tag, M)
    if audio:
        write_audio(models, utts, out / "audio", M)
    print(TABLE, flush=True)
    return RES, TABLE


if globals().get("OV3_RUN_EVAL", True):
    models_ov3, ARMS_OV3 = load_ov3(OV3_TAG)
    print("arms:", {k: (ARMS_OV3[k], models_ov3[k].tau) for k in models_ov3}, flush=True)
    EVAL12 = [np.asarray(u, np.float64) for u in test_utts[:12]]
    RES3, TABLE3 = run_eval(models_ov3, ARMS_OV3, EVAL12, M_DRAWS, OV3_TAG)
    print("EVAL DONE", OV3_TAG, flush=True)

## 15. Perceptual evaluation — LSD and ViSQOL

The user's `evaluation.py` call path: ViSQOL speech mode at 16 kHz, ViSQOL audio mode at 48 kHz (which sees the
whole reconstructed band), and wideband PESQ, for every condition with and without passthrough, plus the floor
(passthrough with an empty high band) and ceiling (passthrough with the true high band) rows. Requires the
ViSQOL install cells to have been run in this kernel.

In [ ]:
OV3_RUN_EVAL = True
OV3_TAG = globals().get("OV3_TAG", "OV3_fast")

# ---- EVALUATION keeps the repo's reproducibility: section 0a turned the strict pair off for TRAINING
# only.  Everything below runs under no_grad (and the decoder's perturb=False path has no scatter at
# all), so determinism is nearly free here.
seed_everything()
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ---- inlined verbatim from overnight3/e5_visqol.py ----
# ============================================================ OV3-5 perceptual evaluation: LSD + ViSQOL (speech 16k, audio 48k) + PESQ
# Every OV3 condition on EVAL12 through the user's evaluation.py, the same call path as overnight2/c6_visqol_eval.py
# (torchaudio resample to 16 kHz, ViSQOL speech mode; ViSQOL audio mode at 48 kHz sees the whole band; PESQ wb).
# ASSUMES the install cells overnight2/c6_install_bg.py, c6_install_wait.py AND c6_lattice.py were run in this
# kernel (visqol-python, pesq, torchmetrics importable; ai_edge_litert for the upstream lattice speech-MOS mapping,
# which the OV2 numbers used) and that evaluation.py sits next to the ov2 / ov3 cells on Drive.  Which speech
# mapping is active is checked below and recorded in the JSON and table (NSIM is mapping-free either way).
# Conditions: naive; per arm one draw (tau=1 seed 0; tau=0 for det); per stochastic arm mean16 and logmean16;
# all of those with and without baseband passthrough; floor (passthrough + empty HB); ceiling (passthrough + true HB).
# Draws are made once per (arm, utterance) and shared by every derived condition.
# Requires: c0 boot, c1_model, e1_model.  Functions at top level; the main block honours OV3_RUN_EVAL.
import json, time, math, numpy as np, torch, importlib.util
from pathlib import Path

OV3_TAG = globals().get("OV3_TAG", "OV3_es")
M_DRAWS = int(globals().get("OV3_M", 16))

def _find_eval_py():
    cands = [Path(str(globals().get("OV3", ""))) / "evaluation.py" if globals().get("OV3") else None,
             Path(str(globals().get("OV2", ""))) / "evaluation.py" if globals().get("OV2") else None,
             ROOT / "evaluation.py", ROOT / "ov3" / "evaluation.py", ROOT / "ov2" / "evaluation.py"]
    for p in cands:
        if p is not None and p.exists():
            return p
    raise FileNotFoundError("evaluation.py not found next to the ov2/ov3 cells or under ROOT")

_spec = importlib.util.spec_from_file_location("user_evaluation", _find_eval_py())
user_eval = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(user_eval)
EVALUATOR = user_eval.Evaluator()                                   # speech mode, 16 kHz, + PESQ wb
try:
    import ai_edge_litert; VISQOL_SP_MAPPING = "lattice"
except Exception:
    VISQOL_SP_MAPPING = "polynomial"
    print("WARNING: speech MOS on the polynomial mapping; run overnight2/c6_lattice.py for OV2-comparable numbers", flush=True)
print("visqol speech16k MOS mapping:", VISQOL_SP_MAPPING, flush=True)
from visqol import VisqolApi
try:
    VQ_AUDIO = VisqolApi(); VQ_AUDIO.create(mode="audio")           # 48 kHz, 32 ERB bands to 24 kHz
except Exception as _e:
    print("audio mode unavailable:", repr(_e)); VQ_AUDIO = None

def visqol_speech(y, w):
    '''MOS-LQO exactly as evaluation.py computes it (torchaudio resample to 16 kHz, speech mode) + NSIM.'''
    yt, wt = torch.from_numpy(np.asarray(y, np.float32)), torch.from_numpy(np.asarray(w, np.float32))
    hr, sr_ = EVALUATOR.sample_to_correct_rate(yt, wt, CFG.fs_hi)
    hr, sr_ = EVALUATOR.match_length(hr, sr_)
    try:
        res = EVALUATOR.visqol_api.measure_from_arrays(hr.numpy().astype(np.float64), sr_.numpy().astype(np.float64), sample_rate=EVALUATOR.target_sr)
        return float(res.moslqo), float(getattr(res, "vnsim", float("nan")))
    except Exception:
        return float("nan"), float("nan")

def pesq_wb(y, w):
    try:
        return float(EVALUATOR.evaluate_pesq(torch.from_numpy(np.asarray(y, np.float32))[None],
                                             torch.from_numpy(np.asarray(w, np.float32))[None], current_sr=CFG.fs_hi))
    except Exception:
        return float("nan")

def visqol_audio(y, w):
    if VQ_AUDIO is None:
        return float("nan"), float("nan")
    n = min(len(y), len(w))
    try:
        res = VQ_AUDIO.measure_from_arrays(np.asarray(y[:n], np.float64), np.asarray(w[:n], np.float64), sample_rate=CFG.fs_hi)
        return float(res.moslqo), float(getattr(res, "vnsim", float("nan")))
    except Exception:
        return float("nan"), float("nan")

def score(y, w):
    y = np.asarray(y, np.float64); w = np.asarray(w, np.float64)[:len(y)]
    w = np.pad(w, (0, len(y) - len(w)))
    vs, ns = visqol_speech(y, w); va, na = visqol_audio(y, w)
    return dict(snr=snr_db(y, w), lsd=lsd_db(y, w, CFG.eval_n_fft, CFG.eval_hop),
                hb_lsd=lsd_db(y, w, CFG.eval_n_fft, CFG.eval_hop, CFG.eval_k_cut),
                visqol_speech16k=vs, nsim_speech16k=ns, pesq_wb=pesq_wb(y, w), visqol_audio48k=va, nsim_audio48k=na)

def _split_bands5(w, fs, f_cut):
    W = np.fft.rfft(np.asarray(w, np.float64))
    k = int(round(f_cut * len(w) / fs))
    lo, hi = W.copy(), W.copy()
    lo[k:] = 0; hi[:k] = 0
    return np.fft.irfft(lo, n=len(w)), np.fft.irfft(hi, n=len(w))

def passthrough5(y, w):
    y = np.asarray(y, np.float64); n = len(y)
    nv = naive_upsample(y, CFG)[:n]; nv = np.pad(nv, (0, n - len(nv)))
    w = np.asarray(w, np.float64)[:n]; w = np.pad(w, (0, n - len(w)))
    return _split_bands5(nv, CFG.fs_hi, CFG.fs_lo / 2)[0] + _split_bands5(w, CFG.fs_hi, CFG.fs_lo / 2)[1]

def conditions_for(models, y, M):
    '''name -> waveform for one utterance; draws made once per arm.'''
    y = np.asarray(y, np.float64)
    out = {"naive": naive_upsample(y, CFG)[:len(y)]}
    for k, m in models.items():
        if m.tau == 0:
            out[k] = reconstruct(m, y, CFG, tau=0.0)
            continue
        draws = np.stack([reconstruct(m, y, CFG, tau=1.0, seed=s) for s in range(M)])
        out[f"{k} tau=1"] = draws[0]
        out[f"{k} mean{M}"] = draws.mean(0)
        out[f"{k} logmean{M}"] = logmag_ensemble_readout(draws, y, CFG, passthrough=False)
    for name in list(out):
        if name != "naive":
            out[f"{name} | passthrough"] = passthrough5(y, out[name])
    out["floor: passthrough + empty HB"] = passthrough5(y, np.zeros_like(y))
    out["ceiling: passthrough + true HB"] = passthrough5(y, y)
    return out

KEYS = ("snr", "lsd", "hb_lsd", "visqol_speech16k", "nsim_speech16k", "pesq_wb", "visqol_audio48k", "nsim_audio48k")

def run_visqol(models, utts, M, tag):
    out_dir = ROOT / "ov3"; out_dir.mkdir(parents=True, exist_ok=True)
    per, t0 = {}, time.time()
    for ui, y in enumerate(utts):
        conds = conditions_for(models, y, M)
        for cname, w in conds.items():
            per.setdefault(cname, []).append(score(y, w))
        print(f"  {ui+1}/{len(utts)} utts, {len(conds)} conditions  [{time.time()-t0:.0f}s]", flush=True)
        json.dump({"per_utt": per, "n": ui + 1, "M": M, "speech_mapping": VISQOL_SP_MAPPING}, open(out_dir / f"visqol_{tag}.json", "w"))
    AGG = {c: {k: float(np.nanmean([r[k] for r in rows])) for k in KEYS} for c, rows in per.items()}
    json.dump({"agg": AGG, "per_utt": per, "n": len(utts), "M": M, "speech_mapping": VISQOL_SP_MAPPING}, open(out_dir / f"visqol_{tag}.json", "w"), indent=1)
    lines = ["| condition | SNR | LSD | HB-LSD | ViSQOL speech16k | NSIM sp | PESQ wb | ViSQOL audio48k | NSIM au |",
             "|---|---|---|---|---|---|---|---|---|"]
    for c in sorted(AGG, key=lambda c: AGG[c]["lsd"]):
        a = AGG[c]
        lines.append(f"| {c} | {a['snr']:.2f} | {a['lsd']:.3f} | {a['hb_lsd']:.3f} | {a['visqol_speech16k']:.3f} | {a['nsim_speech16k']:.3f} | "
                     f"{a['pesq_wb']:.3f} | {a['visqol_audio48k']:.3f} | {a['nsim_audio48k']:.3f} |")
    lines.append(f"\nEVAL12 n={len(utts)}, M={M}; sorted by LSD; speech16k MOS mapping: {VISQOL_SP_MAPPING}"
                 + ("" if VISQOL_SP_MAPPING == "lattice" else " (NOT the lattice mapping of the OV2 tables; compare NSIM)") + ".")
    TABLE = chr(10).join(lines)
    (out_dir / f"visqol_table_{tag}.md").write_text(TABLE)
    print(TABLE, flush=True)
    return AGG, TABLE


if globals().get("OV3_RUN_EVAL", True):
    if "models_ov3" not in globals():
        models_ov3 = {}
        for p in sorted((CKPT / OV3_TAG).glob("*.pt")):
            models_ov3[p.stem], _ = load_arm(p)
    for _m in models_ov3.values():
        _m.eval()
    EVAL12 = [np.asarray(u, np.float64) for u in test_utts[:12]]
    AGG5, TABLE5 = run_visqol(models_ov3, EVAL12, M_DRAWS, OV3_TAG)
    print("VISQOL EVAL DONE", OV3_TAG, flush=True)